# Backpropagation: Complete Manual Study Guide

**DLA Comprehensive Examination Preparation**

---

This notebook teaches backpropagation the way your professor tests it — **by hand, step by step**, with every chain-rule expansion made explicit.

### What you will find here

| Part | Content |
|------|---------|
| **0** | Conceptual foundation — what backprop is and why it works |
| **1** | The chain rule as the engine of backprop |
| **2** | Computation graphs — the visual language of gradients |
| **3** | General equations and notation |
| **4** | **Example 1** — 2-2-1 network, Sigmoid + MSE (full hand derivation) |
| **5** | **Example 2** — 2-2-3 network, ReLU + Softmax + CE (with CE+Softmax derivation) |
| **6** | **Example 3** — 3-3-2 network, Sigmoid + Softmax + CE |
| **7** | Numerical gradient checking |
| **8** | Vanishing gradients |
| **9** | Pitfall warnings |
| **10** | Exercises with solutions |
| **11** | Exam quick reference |

### How to use this notebook

1. Read each markdown section carefully.
2. Try to reproduce every numerical result **before** looking at the code.
3. Attempt each exercise independently. Reveal solutions only after your own attempt.
4. The **Exam Checkpoint** cells summarise what you must be able to write from memory.

> **Professor's note (paraphrased):** Manual backpropagation — knowing every gradient step by hand — is the most critical content of the exam. This notebook trains exactly that skill.


---
## Part 0 — Conceptual Foundation

### What is backpropagation?

A neural network computes a **loss** $L$ that measures how wrong its predictions are. To improve, the network needs to know: *if I nudge each weight slightly, does the loss go up or down, and by how much?*

That "by how much" is the **gradient** $\frac{\partial L}{\partial w}$. Backpropagation is the algorithm that computes all of these gradients efficiently using the **chain rule of calculus**.

### The gradient as a signal

$$
\frac{\partial L}{\partial w} > 0 \implies \text{increasing } w \text{ increases the loss} \implies \text{we should decrease } w
$$

$$
\frac{\partial L}{\partial w} < 0 \implies \text{increasing } w \text{ decreases the loss} \implies \text{we should increase } w
$$

The gradient descent update rule:

$$
\boxed{w \leftarrow w - \eta \frac{\partial L}{\partial w}}
$$

where $\eta > 0$ is the **learning rate**. The negative sign ensures we move in the direction that reduces $L$.

### Why "back" propagation?

Forward pass: data flows **input → hidden → output → loss** (left to right).

Backward pass: gradients flow **loss → output → hidden → input** (right to left).

Each layer receives the gradient from the layer ahead of it and uses the **chain rule** to compute its own gradient before passing a signal further back. This is why it is called *back*propagation.


---
## Part 1 — The Chain Rule: Engine of Backpropagation

### Scalar chain rule

If $y = f(u)$ and $u = g(x)$, so that $y = f(g(x))$, then:

$$
\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx}
$$

**Example:** $y = \sigma(wx + b)$, where $u = wx + b$.

$$
\frac{dy}{dx} = \sigma'(u) \cdot w
$$

### Extended chain rule through a network layer

A single layer computes:

$$
z = w_1 x_1 + w_2 x_2 + b \quad \xrightarrow{\text{activation}} \quad a = g(z) \quad \xrightarrow{\text{loss}} \quad L
$$

Using the chain rule:

$$
\frac{\partial L}{\partial w_1}
= \frac{\partial L}{\partial a} \cdot \frac{\partial a}{\partial z} \cdot \frac{\partial z}{\partial w_1}
= \frac{\partial L}{\partial a} \cdot g'(z) \cdot x_1
$$

Three factors, each with a clear meaning:

| Factor | Notation | Meaning |
|--------|----------|---------|
| $\frac{\partial L}{\partial a}$ | upstream gradient | how much the loss cares about this activation |
| $g'(z)$ | local gradient | how much the activation changes with its input |
| $x_1$ | input | how much the pre-activation changes with the weight |

> **Key insight:** The term $\frac{\partial L}{\partial a} \cdot g'(z)$ appears repeatedly. We name it the **delta** of the layer: $\delta = \frac{\partial L}{\partial z}$. All gradient computations reduce to: find $\delta$, then multiply by the appropriate input.

### Activation function derivatives (memorise these)

| Activation | Formula | Derivative |
|------------|---------|------------|
| Sigmoid | $\sigma(z) = \frac{1}{1+e^{-z}}$ | $\sigma'(z) = \sigma(z)(1-\sigma(z))$ |
| ReLU | $\max(0, z)$ | $1$ if $z > 0$, else $0$ |
| Softmax + CE | $\hat{y}_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$ | See Part 5 for derivation |

> **Pitfall warning — sigmoid derivative:** Always compute $\sigma'$ from the **activation** $a$, not from $z$ directly: $\sigma'(z) = a(1-a)$ where $a = \sigma(z)$. You already have $a$ from the forward pass. Never recompute $\sigma(z)$ from scratch during backprop.


---
## Part 2 — Computation Graphs

A computation graph makes the chain rule visual. Each **node** is an operation; each **edge** carries the local gradient.

### Simple scalar example

Let $f = (x + y) \cdot z$ with $x=2, y=3, z=4$.

**Forward pass** (label each intermediate value):

$$
q = x + y = 5, \qquad f = q \cdot z = 20
$$

**Backward pass** (compute local gradients, multiply by upstream):

$$
\frac{\partial f}{\partial f} = 1 \quad \text{(always)}
$$

$$
\frac{\partial f}{\partial q} = z = 4, \qquad \frac{\partial f}{\partial z} = q = 5
$$

$$
\frac{\partial f}{\partial x} = \frac{\partial f}{\partial q} \cdot \frac{\partial q}{\partial x} = 4 \cdot 1 = 4
$$

$$
\frac{\partial f}{\partial y} = \frac{\partial f}{\partial q} \cdot \frac{\partial q}{\partial y} = 4 \cdot 1 = 4
$$

```
Forward:  x=2 ─┐
               [+]──q=5──[×]──f=20
          y=3 ─┘         │
                        z=4

Backward: ∂f/∂x=4 ─┐
                   [+grad=4]──←──[×grad=1]──←──∂f/∂f=1
          ∂f/∂y=4 ─┘               ↑
                               ∂f/∂z=5
```

### Neural network as a computation graph

For a two-layer network, the forward pass graph is:

```
x ──[W¹,b¹]──> z¹ ──[g(.)]──> a¹ ──[W²,b²]──> z² ──[σ(.)]──> ŷ ──[Loss]──> L
```

The backward pass reverses every arrow:

```
∂L/∂W¹ <──δ¹──[g'(z¹)]<──[W²ᵀ]──δ² <──[g'(z²)]<──[ŷ−y]<── L
```

This is the entire structure of backpropagation. Everything else is substituting specific activation functions.


---
## Part 3 — General Layer Equations

These four equations describe backpropagation for any feedforward network with $L$ layers.

### Forward pass (layer $\ell$)

$$
z^{[\ell]} = W^{[\ell]} a^{[\ell-1]} + b^{[\ell]}
$$

$$
a^{[\ell]} = g^{[\ell]}(z^{[\ell]})
$$

where $a^{[0]} = x$ (the input).

### Backward pass (layer $\ell$)

**Output layer** ($\ell = L$):

$$
\delta^{[L]} = \frac{\partial L}{\partial a^{[L]}} \odot g'^{[L]}(z^{[L]})
$$

**Hidden layer** ($\ell < L$):

$$
\boxed{\delta^{[\ell]} = \left(W^{[\ell+1]}\right)^T \delta^{[\ell+1]} \odot g'^{[\ell]}(z^{[\ell]})}
$$

**Gradients** (same for every layer):

$$
\boxed{\frac{\partial L}{\partial W^{[\ell]}} = \delta^{[\ell]} \left(a^{[\ell-1]}\right)^T}
$$

$$
\boxed{\frac{\partial L}{\partial b^{[\ell]}} = \delta^{[\ell]}}
$$

### Symbol $\odot$: the Hadamard product

$\odot$ is **element-wise multiplication** (not matrix multiplication). This symbol is one of the most common sources of exam errors.

$$
\begin{bmatrix}a \\ b\end{bmatrix} \odot \begin{bmatrix}c \\ d\end{bmatrix} = \begin{bmatrix}ac \\ bd\end{bmatrix}
$$

Compare with matrix multiplication: $\begin{bmatrix}a & b\end{bmatrix}\begin{bmatrix}c \\ d\end{bmatrix} = ac+bd$ (a scalar).

> **Why $\odot$?** $\delta^{[\ell]}$ is the gradient of $L$ with respect to $z^{[\ell]}$. Since $a^{[\ell]} = g(z^{[\ell]})$ element-wise, the chain rule applies element-wise — which is precisely the Hadamard product.

### Special case: Softmax + Cross-Entropy

For the combination softmax output + cross-entropy loss + one-hot target:

$$
\delta^{[L]} = \hat{y} - y
$$

This is not obvious. The derivation is in Part 5 (Example 2).


In [ ]:
import numpy as np
np.set_printoptions(precision=6, suppress=True)

# ── Activation functions ──────────────────────────────────────────────────
def sigmoid(z):
    """Numerically stable sigmoid."""
    return 1.0 / (1.0 + np.exp(-z))

def sigmoid_d(a):
    """Sigmoid derivative from activation (not from z).
    Uses the identity σ'(z) = σ(z)(1 − σ(z)) = a(1 − a).
    Always compute from `a` (already calculated in the forward pass).
    """
    return a * (1.0 - a)

def relu(z):
    return np.maximum(0.0, z)

def relu_d(z):
    """ReLU derivative: 1 if z > 0, else 0.
    Note: applied to z (pre-activation), not a.
    """
    return (z > 0).astype(float)

def softmax(z):
    """Numerically stable softmax: subtract max before exponentiation.
    This prevents overflow for large z without changing the result,
    because softmax is invariant to constant shifts:
    e^(z_i - C) / Σ e^(z_j - C) = e^z_i / Σ e^z_j.
    """
    z_shift = z - np.max(z)        # shift by max for numerical stability
    exp_z   = np.exp(z_shift)
    return exp_z / np.sum(exp_z)

def mse_loss(y_hat, y):
    """½Σ(ŷ − y)² — the ½ makes the gradient cleaner."""
    return 0.5 * np.sum((y_hat - y) ** 2)

def cross_entropy(y_hat, y):
    """−Σ y_i log(ŷ_i).  Clip for numerical safety."""
    return -np.sum(y * np.log(np.clip(y_hat, 1e-12, 1.0)))

print("All helper functions loaded.")


All helper functions loaded.


---
## Part 4 — Example 1: 2-2-1 Network, Sigmoid + MSE

### Architecture

```
         Hidden layer          Output layer
                      w₁[²]
x₁ ─w₁₁─┐           ╱          ╲
         h₁─sigmoid─            output─sigmoid─ ŷ ──> Loss (MSE)
x₂ ─w₁₂─┘    (a₁)   ╲          ╱
                      w₂[²]
         h₂─sigmoid─
x₁ ─w₂₁─┘    (a₂)
x₂ ─w₂₂─┘
```

This example uses:
- 2 input neurons, 2 hidden neurons (sigmoid), 1 output neuron (sigmoid)
- Mean Squared Error loss: $L = \frac{1}{2}(\hat{y} - y)^2$

### Given parameters

**Input and target:**

$$
x_1 = 1,\quad x_2 = 0,\quad y = 1
$$

**Hidden layer weights and biases:**

$$
w_{11}^{[1]} = 0.1,\quad w_{12}^{[1]} = 0.2,\quad b_1^{[1]} = 0.1
$$

$$
w_{21}^{[1]} = 0.3,\quad w_{22}^{[1]} = 0.4,\quad b_2^{[1]} = 0.1
$$

**Output layer weights and bias:**

$$
w_1^{[2]} = 0.5,\quad w_2^{[2]} = 0.6,\quad b^{[2]} = 0.1
$$


### Forward Pass — Step by Step

**Step 1: Hidden neuron pre-activations**

$$
z_1^{[1]} = w_{11}^{[1]} x_1 + w_{12}^{[1]} x_2 + b_1^{[1]}
= (0.1)(1) + (0.2)(0) + 0.1 = 0.2
$$

$$
z_2^{[1]} = w_{21}^{[1]} x_1 + w_{22}^{[1]} x_2 + b_2^{[1]}
= (0.3)(1) + (0.4)(0) + 0.1 = 0.4
$$

**Step 2: Hidden neuron activations** $a = \sigma(z) = \frac{1}{1+e^{-z}}$

$$
a_1^{[1]} = \sigma(0.2) = \frac{1}{1+e^{-0.2}} \approx 0.5498
$$

$$
a_2^{[1]} = \sigma(0.4) = \frac{1}{1+e^{-0.4}} \approx 0.5987
$$

**Step 3: Output pre-activation**

$$
z^{[2]} = w_1^{[2]} a_1^{[1]} + w_2^{[2]} a_2^{[1]} + b^{[2]}
= (0.5)(0.5498) + (0.6)(0.5987) + 0.1
= 0.2749 + 0.3592 + 0.1 = 0.7341
$$

**Step 4: Output activation**

$$
\hat{y} = \sigma(0.7341) \approx 0.6757
$$

**Step 5: Loss**

$$
L = \frac{1}{2}(\hat{y} - y)^2 = \frac{1}{2}(0.6757 - 1)^2 = \frac{1}{2}(0.1052) \approx 0.0526
$$


### Backward Pass — Step by Step

We work **right to left**. At each step, we expand the chain rule fully before substituting numbers.

---

**Step 1: Gradient of loss with respect to $\hat{y}$**

$$
\frac{\partial L}{\partial \hat{y}}
= \frac{\partial}{\partial \hat{y}}\left[\frac{1}{2}(\hat{y}-y)^2\right]
= \hat{y} - y
= 0.6757 - 1 = -0.3243
$$

---

**Step 2: Output delta $\delta^{[2]}$**

$$
\delta^{[2]}
= \underbrace{\frac{\partial L}{\partial \hat{y}}}_{\text{upstream}} \cdot \underbrace{\frac{\partial \hat{y}}{\partial z^{[2]}}}_{\text{local}} \quad \leftarrow \text{chain rule}
$$

The local gradient is the sigmoid derivative:

$$
\frac{\partial \hat{y}}{\partial z^{[2]}} = \hat{y}(1-\hat{y}) = (0.6757)(0.3243) \approx 0.2191
$$

Therefore:

$$
\delta^{[2]} = (-0.3243)(0.2191) \approx -0.0710
$$

---

**Step 3: Output-layer weight gradients**

Since $z^{[2]} = w_1^{[2]} a_1^{[1]} + w_2^{[2]} a_2^{[1]} + b^{[2]}$, applying the chain rule:

$$
\frac{\partial L}{\partial w_1^{[2]}}
= \underbrace{\delta^{[2]}}_{\partial L/\partial z^{[2]}} \cdot \underbrace{\frac{\partial z^{[2]}}{\partial w_1^{[2]}}}_{= a_1^{[1]}}
= (-0.0710)(0.5498) \approx -0.0390
$$

$$
\frac{\partial L}{\partial w_2^{[2]}}
= \delta^{[2]} \cdot a_2^{[1]}
= (-0.0710)(0.5987) \approx -0.0425
$$

$$
\frac{\partial L}{\partial b^{[2]}}
= \delta^{[2]} \cdot 1 = -0.0710
$$

> **Pattern:** Every weight gradient = (delta of this layer) × (activation feeding into this weight). The bias gradient = delta directly.

---

**Step 4: Hidden delta $\delta_1^{[1]}$**

The gradient must travel from $\delta^{[2]}$ back through the weight $w_1^{[2]}$ and then through the sigmoid at $h_1$:

$$
\delta_1^{[1]}
= \underbrace{\delta^{[2]} \cdot \frac{\partial z^{[2]}}{\partial a_1^{[1]}}}_{\text{gradient reaching }a_1^{[1]}} \cdot \underbrace{\frac{\partial a_1^{[1]}}{\partial z_1^{[1]}}}_{\text{sigmoid derivative}}
$$

$$
\frac{\partial z^{[2]}}{\partial a_1^{[1]}} = w_1^{[2]} = 0.5
$$

$$
\frac{\partial a_1^{[1]}}{\partial z_1^{[1]}} = a_1^{[1]}(1-a_1^{[1]}) = (0.5498)(0.4502) \approx 0.2475
$$

$$
\delta_1^{[1]} = (-0.0710)(0.5)(0.2475) \approx -0.0088
$$

---

**Step 5: Hidden delta $\delta_2^{[1]}$**

Same structure, but through $w_2^{[2]}$ and the sigmoid at $h_2$:

$$
\delta_2^{[1]}
= \delta^{[2]} \cdot w_2^{[2]} \cdot a_2^{[1]}(1-a_2^{[1]})
$$

$$
a_2^{[1]}(1-a_2^{[1]}) = (0.5987)(0.4013) \approx 0.2403
$$

$$
\delta_2^{[1]} = (-0.0710)(0.6)(0.2403) \approx -0.0102
$$

---

**Step 6: Hidden-layer weight gradients**

$$
\frac{\partial L}{\partial w_{11}^{[1]}} = \delta_1^{[1]} \cdot x_1 = (-0.0088)(1) = -0.0088
$$

$$
\frac{\partial L}{\partial w_{12}^{[1]}} = \delta_1^{[1]} \cdot x_2 = (-0.0088)(0) = 0
$$

$$
\frac{\partial L}{\partial b_1^{[1]}} = \delta_1^{[1]} = -0.0088
$$

$$
\frac{\partial L}{\partial w_{21}^{[1]}} = \delta_2^{[1]} \cdot x_1 = (-0.0102)(1) = -0.0102
$$

$$
\frac{\partial L}{\partial w_{22}^{[1]}} = \delta_2^{[1]} \cdot x_2 = (-0.0102)(0) = 0
$$

$$
\frac{\partial L}{\partial b_2^{[1]}} = \delta_2^{[1]} = -0.0102
$$

> **Note:** $w_{12}^{[1]}$ and $w_{22}^{[1]}$ gradients are both zero because $x_2 = 0$. The gradient tells us that these weights had no influence on this forward pass. This is correct, not a bug.

---

### Final Gradient Summary — Example 1

| Parameter | Gradient |
|-----------|---------|
| $w_1^{[2]}$ | $-0.0390$ |
| $w_2^{[2]}$ | $-0.0425$ |
| $b^{[2]}$ | $-0.0710$ |
| $w_{11}^{[1]}$ | $-0.0088$ |
| $w_{12}^{[1]}$ | $0$ |
| $b_1^{[1]}$ | $-0.0088$ |
| $w_{21}^{[1]}$ | $-0.0102$ |
| $w_{22}^{[1]}$ | $0$ |
| $b_2^{[1]}$ | $-0.0102$ |


In [ ]:
# ── Example 1: NumPy implementation ──────────────────────────────────────
# Variable names map directly to the mathematical notation above.
# W1[i,j] = w^[1]_{ij}, delta2 = δ^[2], dW2 = ∂L/∂W^[2], etc.

x  = np.array([[1.0], [0.0]])  # shape (2,1)
y  = np.array([[1.0]])          # shape (1,1)

W1 = np.array([[0.1, 0.2],
               [0.3, 0.4]])    # shape (2,2): W1[i,j] = w^[1]_{i+1, j+1}
b1 = np.array([[0.1], [0.1]])  # shape (2,1)
W2 = np.array([[0.5, 0.6]])    # shape (1,2): [w_1^[2], w_2^[2]]
b2 = np.array([[0.1]])          # shape (1,1)

# ── Forward pass ──────────────────────────────────────────────────────────
z1    = W1 @ x + b1            # (2,1) pre-activations of hidden layer
a1    = sigmoid(z1)            # (2,1) activations of hidden layer
z2    = W2 @ a1 + b2           # (1,1) pre-activation of output
y_hat = sigmoid(z2)            # (1,1) output prediction
L     = mse_loss(y_hat, y)     # scalar loss

print("─── Example 1: Forward Pass ───")
print(f"z1      = {z1.ravel()}")       # should be [0.2, 0.4]
print(f"a1      = {a1.ravel()}")       # should be [0.5498, 0.5987]
print(f"z2      = {z2.ravel()}")       # should be [0.7341]
print(f"y_hat   = {y_hat.ravel()}")    # should be [0.6757]
print(f"Loss    = {L:.6f}")            # should be 0.0526


─── Example 1: Forward Pass ───
z1      = [0.2 0.4]
a1      = [0.549834 0.598688]
z2      = [0.73413]
y_hat   = [0.675711]
Loss    = 0.052582


In [ ]:
# ── Backward pass ────────────────────────────────────────────────────────
# Step 1+2: Output delta  δ^[2] = (ŷ - y) ⊙ σ'(z^[2])
#           Since σ'(z) = a(1-a), we use sigmoid_d(y_hat).
dL_dyhat = y_hat - y                          # ∂L/∂ŷ = ŷ − y
delta2   = dL_dyhat * sigmoid_d(y_hat)        # δ^[2] = ∂L/∂z^[2]   shape (1,1)

# Step 3: Output-layer gradients
dW2 = delta2 @ a1.T   # ∂L/∂W^[2] = δ^[2] · (a^[1])ᵀ  shape (1,2)
db2 = delta2           # ∂L/∂b^[2] = δ^[2]              shape (1,1)

# Step 4+5: Hidden deltas   δ^[1] = (W^[2])ᵀ δ^[2] ⊙ σ'(z^[1])
#           (W^[2])ᵀ carries the gradient back through the weights.
#           ⊙ σ'(z^[1]) applies the local sigmoid derivative.
delta1 = (W2.T @ delta2) * sigmoid_d(a1)     # shape (2,1)

# Step 6: Hidden-layer gradients
dW1 = delta1 @ x.T    # ∂L/∂W^[1] = δ^[1] · xᵀ  shape (2,2)
db1 = delta1           # ∂L/∂b^[1] = δ^[1]         shape (2,1)

print("─── Example 1: Backward Pass ───")
print(f"dL/dŷ   = {dL_dyhat.ravel()}")   # [-0.3243]
print(f"delta2  = {delta2.ravel()}")      # [-0.0710]
print(f"dW2     = {dW2.ravel()}")         # [-0.0390, -0.0425]
print(f"db2     = {db2.ravel()}")         # [-0.0710]
print(f"delta1  = {delta1.ravel()}")      # [-0.0088, -0.0102]
print(f"dW1     =\n{dW1}")               # rows: hidden neuron 1 and 2
print(f"db1     = {db1.ravel()}")         # [-0.0088, -0.0102]


─── Example 1: Backward Pass ───
dL/dŷ   = [-0.324289]
delta2  = [-0.07106]
dW2     = [-0.039071 -0.042543]
db2     = [-0.07106]
delta1  = [-0.008794 -0.010244]
dW1     =
[[-0.008794  0.      ]
 [-0.010244  0.      ]]
db1     = [-0.008794 -0.010244]


### Exam Checkpoint 1 ✓

> **Write these from memory before moving on.**

1. $\delta^{[2]} = (\hat{y} - y) \cdot \hat{y}(1-\hat{y})$ for MSE + sigmoid output
2. $\delta^{[\ell]} = (W^{[\ell+1]})^T \delta^{[\ell+1]} \odot g'(z^{[\ell]})$ for any hidden layer
3. $\frac{\partial L}{\partial W^{[\ell]}} = \delta^{[\ell]} (a^{[\ell-1]})^T$ — **the ${}^T$ goes on $a$, not on $\delta$**
4. $\frac{\partial L}{\partial b^{[\ell]}} = \delta^{[\ell]}$ (no extra multiplication needed)
5. A weight gradient is zero if its input $x_j = 0$ — correct, not a bug


---
## Part 5 — Example 2: 2-2-3 Network, ReLU + Softmax + Cross-Entropy

### Why this combination matters

ReLU hidden + Softmax output + Cross-Entropy loss is the **standard architecture for classification**. It appears in virtually every DLA exam and every real classifier. The output-layer gradient simplifies beautifully — but only if you know why.

---

### Derivation: Why $\delta^{[L]} = \hat{y} - y$ for Softmax + CE

**This is the derivation professors test.** Memorising the result without the derivation is insufficient for a comps exam.

#### Setup

Softmax output for class $i$ (across $K$ classes):

$$
\hat{y}_i = \frac{e^{z_i}}{\sum_{k=1}^{K} e^{z_k}}
$$

Cross-entropy loss with one-hot target $y$:

$$
L = -\sum_{i=1}^{K} y_i \log \hat{y}_i
$$

We want $\delta_i^{[L]} = \frac{\partial L}{\partial z_i}$ for each output unit $i$.

#### Step 1: Gradient of $L$ with respect to $\hat{y}_i$

$$
\frac{\partial L}{\partial \hat{y}_i} = -\frac{y_i}{\hat{y}_i}
$$

#### Step 2: Gradient of $\hat{y}_i$ with respect to $z_j$ (the Jacobian)

The softmax Jacobian has two cases:

$$
\frac{\partial \hat{y}_i}{\partial z_j} =
\begin{cases}
\hat{y}_i(1 - \hat{y}_i) & \text{if } i = j \quad \text{(on-diagonal)}\\
-\hat{y}_i \hat{y}_j & \text{if } i \neq j \quad \text{(off-diagonal)}
\end{cases}
$$

This can be written compactly as $\hat{y}_i(\delta_{ij} - \hat{y}_j)$, where $\delta_{ij}$ is the Kronecker delta.

#### Step 3: Apply the chain rule

$$
\frac{\partial L}{\partial z_j}
= \sum_{i=1}^{K} \frac{\partial L}{\partial \hat{y}_i} \cdot \frac{\partial \hat{y}_i}{\partial z_j}
= \sum_{i} \left(-\frac{y_i}{\hat{y}_i}\right) \hat{y}_i (\delta_{ij} - \hat{y}_j)
$$

$$
= \sum_{i} (-y_i)(\delta_{ij} - \hat{y}_j)
= -\sum_{i} y_i \delta_{ij} + \hat{y}_j \sum_{i} y_i
$$

Since $y$ is one-hot: $\sum_i y_i = 1$ and $\sum_i y_i \delta_{ij} = y_j$. Therefore:

$$
\frac{\partial L}{\partial z_j} = -y_j + \hat{y}_j = \hat{y}_j - y_j
$$

#### Result (vectorised)

$$
\boxed{\delta^{[L]} = \frac{\partial L}{\partial z^{[L]}} = \hat{y} - y}
$$

The Jacobian complexity collapses entirely because the CE and softmax derivatives cancel. This is **only valid** for the combination softmax + cross-entropy + one-hot labels. Do not apply it with other activations or other losses.


### Architecture

```
x₁, x₂  ──> 2 hidden ReLU units ──> 3 output softmax units ──> CE loss
```

### Given parameters

$$
x = \begin{bmatrix}1 \\ 2\end{bmatrix}, \qquad y = \begin{bmatrix}0 \\ 1 \\ 0\end{bmatrix} \quad \text{(true class is class 2)}
$$

$$
W^{[1]} = \begin{bmatrix}0.1 & 0.2 \\ 0.3 & 0.4\end{bmatrix}, \quad b^{[1]} = \begin{bmatrix}0.1 \\ 0.1\end{bmatrix}
$$

$$
W^{[2]} = \begin{bmatrix}0.5 & 0.1 \\ 0.2 & 0.3 \\ 0.4 & 0.6\end{bmatrix}, \quad b^{[2]} = \begin{bmatrix}0.1 \\ 0.1 \\ 0.1\end{bmatrix}
$$

---

### Forward Pass

**Hidden pre-activations:**

$$
z_1^{[1]} = 0.1(1) + 0.2(2) + 0.1 = 0.6
$$

$$
z_2^{[1]} = 0.3(1) + 0.4(2) + 0.1 = 1.2
$$

**Hidden activations (ReLU):** Since both values are positive, ReLU passes them unchanged:

$$
a_1^{[1]} = \text{ReLU}(0.6) = 0.6, \qquad a_2^{[1]} = \text{ReLU}(1.2) = 1.2
$$

**Output logits:**

$$
z_1^{[2]} = (0.5)(0.6) + (0.1)(1.2) + 0.1 = 0.30 + 0.12 + 0.10 = 0.52
$$

$$
z_2^{[2]} = (0.2)(0.6) + (0.3)(1.2) + 0.1 = 0.12 + 0.36 + 0.10 = 0.58
$$

$$
z_3^{[2]} = (0.4)(0.6) + (0.6)(1.2) + 0.1 = 0.24 + 0.72 + 0.10 = 1.06
$$

**Softmax probabilities** (using numerical stability: subtract $\max = 1.06$):

$$
z^{[2]} - 1.06 = \begin{bmatrix}-0.54 \\ -0.48 \\ 0\end{bmatrix}
\implies
e^{\cdot} = \begin{bmatrix}0.5827 \\ 0.6188 \\ 1.0000\end{bmatrix}
$$

$$
S = 0.5827 + 0.6188 + 1.0000 = 2.2015
$$

$$
\hat{y} = \begin{bmatrix}0.2647 \\ 0.2811 \\ 0.4542\end{bmatrix}
$$

**Cross-entropy loss** (true class is class 2, so $y_2 = 1$):

$$
L = -\log(\hat{y}_2) = -\log(0.2811) \approx 1.269
$$

---

### Backward Pass

**Output delta** (using the derived shortcut — valid for softmax + CE + one-hot):

$$
\delta^{[2]} = \hat{y} - y = \begin{bmatrix}0.2647 \\ 0.2811 \\ 0.4542\end{bmatrix} - \begin{bmatrix}0 \\ 1 \\ 0\end{bmatrix} = \begin{bmatrix}0.2647 \\ -0.7189 \\ 0.4542\end{bmatrix}
$$

> **Intuition:** Class 2 (true class) has $\delta_2 = 0.2811 - 1 = -0.7189$. A large negative value signals that the logit for class 2 needs to increase a lot. Wrong classes have positive deltas — their logits need to decrease.

**Output-layer gradients:**

$$
\frac{\partial L}{\partial W^{[2]}} = \delta^{[2]}(a^{[1]})^T =
\begin{bmatrix}0.2647 \\ -0.7189 \\ 0.4542\end{bmatrix}
\begin{bmatrix}0.6 & 1.2\end{bmatrix}
=
\begin{bmatrix}0.1588 & 0.3176 \\ -0.4313 & -0.8627 \\ 0.2725 & 0.5450\end{bmatrix}
$$

$$
\frac{\partial L}{\partial b^{[2]}} = \delta^{[2]} = \begin{bmatrix}0.2647 \\ -0.7189 \\ 0.4542\end{bmatrix}
$$

**Gradient reaching hidden activations:**

$$
\frac{\partial L}{\partial a^{[1]}} = (W^{[2]})^T \delta^{[2]}
= \begin{bmatrix}0.5 & 0.2 & 0.4 \\ 0.1 & 0.3 & 0.6\end{bmatrix}
\begin{bmatrix}0.2647 \\ -0.7189 \\ 0.4542\end{bmatrix}
$$

$$
= \begin{bmatrix}
(0.5)(0.2647) + (0.2)(-0.7189) + (0.4)(0.4542) \\
(0.1)(0.2647) + (0.3)(-0.7189) + (0.6)(0.4542)
\end{bmatrix}
= \begin{bmatrix}0.1703 \\ 0.0833\end{bmatrix}
$$

**Hidden deltas (through ReLU derivative):**

$$
z^{[1]} = \begin{bmatrix}0.6 \\ 1.2\end{bmatrix} \implies \text{ReLU}'(z^{[1]}) = \begin{bmatrix}1 \\ 1\end{bmatrix}
$$

Both pre-activations are positive, so ReLU passes gradients through unchanged:

$$
\delta^{[1]} = \frac{\partial L}{\partial a^{[1]}} \odot \text{ReLU}'(z^{[1]}) = \begin{bmatrix}0.1703 \\ 0.0833\end{bmatrix} \odot \begin{bmatrix}1 \\ 1\end{bmatrix} = \begin{bmatrix}0.1703 \\ 0.0833\end{bmatrix}
$$

**Hidden-layer gradients:**

$$
\frac{\partial L}{\partial W^{[1]}} = \delta^{[1]} x^T = \begin{bmatrix}0.1703 \\ 0.0833\end{bmatrix}\begin{bmatrix}1 & 2\end{bmatrix}
= \begin{bmatrix}0.1703 & 0.3405 \\ 0.0833 & 0.1666\end{bmatrix}
$$

$$
\frac{\partial L}{\partial b^{[1]}} = \delta^{[1]} = \begin{bmatrix}0.1703 \\ 0.0833\end{bmatrix}
$$

> **ReLU pitfall:** If any $z_j^{[1]}$ were $\leq 0$, its ReLU derivative would be 0 and the gradient to that unit would be **blocked completely**. This is the "dying ReLU" phenomenon — important for DLA exams.


In [ ]:
# ── Example 2: NumPy implementation ──────────────────────────────────────

x2 = np.array([[1.0], [2.0]])
y2 = np.array([[0.0], [1.0], [0.0]])

W1b = np.array([[0.1, 0.2], [0.3, 0.4]])
b1b = np.array([[0.1], [0.1]])
W2b = np.array([[0.5, 0.1], [0.2, 0.3], [0.4, 0.6]])
b2b = np.array([[0.1], [0.1], [0.1]])

# Forward
z1b   = W1b @ x2 + b1b
a1b   = relu(z1b)
z2b   = W2b @ a1b + b2b
yhatb = softmax(z2b)          # shape (3,1) — softmax applied column-wise
Lb    = cross_entropy(yhatb, y2)

print("─── Example 2: Forward Pass ───")
print(f"z1      = {z1b.ravel()}")     # [0.6, 1.2]
print(f"a1      = {a1b.ravel()}")     # [0.6, 1.2]  (ReLU leaves positive unchanged)
print(f"z2      = {z2b.ravel()}")     # [0.52, 0.58, 1.06]
print(f"y_hat   = {yhatb.ravel()}")   # [0.2647, 0.2811, 0.4542]
print(f"Loss    = {Lb:.6f}")          # 1.2692

# Backward
# Output delta: the clean softmax+CE shortcut (derived above)
delta2b  = yhatb - y2                         # δ^[2] = ŷ - y   shape (3,1)

dW2b = delta2b @ a1b.T                        # ∂L/∂W^[2]  shape (3,2)
db2b = delta2b                                 # ∂L/∂b^[2]  shape (3,1)

dL_da1b  = W2b.T @ delta2b                    # gradient at a^[1]  shape (2,1)
delta1b  = dL_da1b * relu_d(z1b)              # ⊙ ReLU'(z^[1])
# Note: relu_d takes z (pre-activation), NOT a

dW1b = delta1b @ x2.T                         # ∂L/∂W^[1]  shape (2,2)
db1b = delta1b                                 # ∂L/∂b^[1]  shape (2,1)

print("\n─── Example 2: Backward Pass ───")
print(f"delta2  = {delta2b.ravel()}")  # [ 0.2647, -0.7189,  0.4542]
print(f"dW2     =\n{dW2b}")
print(f"db2     = {db2b.ravel()}")
print(f"dL/da1  = {dL_da1b.ravel()}")  # [0.1703, 0.0833]
print(f"delta1  = {delta1b.ravel()}")  # same (both ReLU derivs = 1)
print(f"dW1     =\n{dW1b}")
print(f"db1     = {db1b.ravel()}")


─── Example 2: Forward Pass ───
z1      = [0.6 1.2]
a1      = [0.6 1.2]
z2      = [0.52 0.58 1.06]
y_hat   = [0.264701 0.281069 0.454229]
Loss    = 1.269153

─── Example 2: Backward Pass ───
delta2  = [ 0.264701 -0.718931  0.454229]
dW2     =
[[ 0.158821  0.317642]
 [-0.431358 -0.862717]
 [ 0.272538  0.545075]]
db2     = [ 0.264701 -0.718931  0.454229]
dL/da1  = [0.170256 0.083329]
delta1  = [0.170256 0.083329]
dW1     =
[[0.170256 0.340512]
 [0.083329 0.166657]]
db1     = [0.170256 0.083329]


### Exam Checkpoint 2 ✓

1. $\delta^{[L]} = \hat{y} - y$ **only** for softmax + cross-entropy + one-hot labels. Know the derivation.
2. The Jacobian cancellation: $\sum_i (-y_i)(\delta_{ij} - \hat{y}_j) = \hat{y}_j - y_j$.
3. ReLU derivative applies to $z$ (pre-activation), not $a$. Sigmoid derivative applies to $a$ (activation).
4. Dying ReLU: if $z \leq 0$, the ReLU derivative is 0 and no gradient passes through that unit.
5. Numerical stability of softmax: subtract max before exponentiation. The result is invariant.


---
## Part 6 — Example 3: 3-3-2 Network, Sigmoid Hidden + Softmax Output + CE

This is a larger exam-style network. It combines sigmoid hidden units with a softmax output — a realistic classification setup.

### Architecture

```
x₁, x₂, x₃  ──> 3 hidden sigmoid units ──> 2 softmax output units ──> CE loss
```

### Given parameters

$$
x = \begin{bmatrix}1 \\ 0 \\ 1\end{bmatrix}, \qquad y = \begin{bmatrix}1 \\ 0\end{bmatrix}
$$

$$
W^{[1]} = \begin{bmatrix}0.10 & 0.20 & 0.30 \\ 0.40 & 0.50 & 0.60 \\ 0.70 & 0.80 & 0.90\end{bmatrix}, \quad b^{[1]} = \begin{bmatrix}0.10 \\ 0.10 \\ 0.10\end{bmatrix}
$$

$$
W^{[2]} = \begin{bmatrix}0.20 & 0.30 & 0.40 \\ 0.50 & 0.60 & 0.70\end{bmatrix}, \quad b^{[2]} = \begin{bmatrix}0.10 \\ 0.10\end{bmatrix}
$$

---

### Forward Pass

**Hidden pre-activations:**

$$
z_1^{[1]} = (0.10)(1)+(0.20)(0)+(0.30)(1)+0.10 = 0.50
$$

$$
z_2^{[1]} = (0.40)(1)+(0.50)(0)+(0.60)(1)+0.10 = 1.10
$$

$$
z_3^{[1]} = (0.70)(1)+(0.80)(0)+(0.90)(1)+0.10 = 1.70
$$

**Hidden activations (sigmoid):**

$$
a_1^{[1]} = \sigma(0.50) \approx 0.6225, \quad a_2^{[1]} = \sigma(1.10) \approx 0.7503, \quad a_3^{[1]} = \sigma(1.70) \approx 0.8455
$$

**Output logits:**

$$
z_1^{[2]} = (0.20)(0.6225)+(0.30)(0.7503)+(0.40)(0.8455)+0.10
= 0.1245+0.2251+0.3382+0.10 \approx 0.7878
$$

$$
z_2^{[2]} = (0.50)(0.6225)+(0.60)(0.7503)+(0.70)(0.8455)+0.10
= 0.3112+0.4502+0.5919+0.10 \approx 1.4533
$$

**Softmax (subtract max = 1.4533 for stability):**

$$
e^{0.7878-1.4533} = e^{-0.6655} \approx 0.5141, \quad e^{0} = 1.0000
$$

$$
\hat{y} = \frac{1}{1.5141}\begin{bmatrix}0.5141 \\ 1.0000\end{bmatrix} \approx \begin{bmatrix}0.3395 \\ 0.6605\end{bmatrix}
$$

**Loss (true class is class 1):**

$$
L = -\log(0.3395) \approx 1.0803
$$

---

### Backward Pass

**Output delta** (softmax + CE shortcut applies):

$$
\delta^{[2]} = \hat{y} - y = \begin{bmatrix}0.3395 \\ 0.6605\end{bmatrix} - \begin{bmatrix}1 \\ 0\end{bmatrix} = \begin{bmatrix}-0.6605 \\ 0.6605\end{bmatrix}
$$

**Output-layer gradients:**

$$
\frac{\partial L}{\partial W^{[2]}} = \delta^{[2]}(a^{[1]})^T = \begin{bmatrix}-0.6605 \\ 0.6605\end{bmatrix}\begin{bmatrix}0.6225 & 0.7503 & 0.8455\end{bmatrix}
$$

$$
= \begin{bmatrix}-0.4112 & -0.4956 & -0.5585 \\ 0.4112 & 0.4956 & 0.5585\end{bmatrix}
$$

$$
\frac{\partial L}{\partial b^{[2]}} = \begin{bmatrix}-0.6605 \\ 0.6605\end{bmatrix}
$$

**Gradient reaching hidden activations:**

$$
\frac{\partial L}{\partial a^{[1]}} = (W^{[2]})^T \delta^{[2]}
= \begin{bmatrix}0.20 & 0.50 \\ 0.30 & 0.60 \\ 0.40 & 0.70\end{bmatrix}\begin{bmatrix}-0.6605 \\ 0.6605\end{bmatrix}
$$

$$
= \begin{bmatrix}(0.20)(-0.6605)+(0.50)(0.6605) \\ (0.30)(-0.6605)+(0.60)(0.6605) \\ (0.40)(-0.6605)+(0.70)(0.6605)\end{bmatrix}
= \begin{bmatrix}0.6605(-0.20+0.50) \\ 0.6605(-0.30+0.60) \\ 0.6605(-0.40+0.70)\end{bmatrix}
= \begin{bmatrix}0.1982 \\ 0.1982 \\ 0.1982\end{bmatrix}
$$

> **Important note — this equal-value result is a coincidence of the chosen parameters, not a general rule.** The specific $W^{[2]}$ used here has column differences that happen to be constant (each column of $W^{[2]}$ increases by 0.10 per row, and $\delta^{[2]}$ sums to zero). In a real network, hidden gradients will almost never all be equal. Do not assume uniform hidden gradients on an exam.

**Sigmoid derivatives for hidden units:**

$$
\sigma'(z) = a(1-a)
$$

$$
\sigma'(0.50) = 0.6225 \times 0.3775 \approx 0.2350
$$

$$
\sigma'(1.10) = 0.7503 \times 0.2497 \approx 0.1874
$$

$$
\sigma'(1.70) = 0.8455 \times 0.1545 \approx 0.1306
$$

**Hidden deltas:**

$$
\delta^{[1]} = \begin{bmatrix}0.1982 \\ 0.1982 \\ 0.1982\end{bmatrix} \odot \begin{bmatrix}0.2350 \\ 0.1874 \\ 0.1306\end{bmatrix} = \begin{bmatrix}0.0466 \\ 0.0372 \\ 0.0259\end{bmatrix}
$$

**Hidden-layer gradients:**

$$
\frac{\partial L}{\partial W^{[1]}} = \delta^{[1]} x^T = \begin{bmatrix}0.0466 \\ 0.0372 \\ 0.0259\end{bmatrix}\begin{bmatrix}1 & 0 & 1\end{bmatrix} = \begin{bmatrix}0.0466 & 0 & 0.0466 \\ 0.0372 & 0 & 0.0372 \\ 0.0259 & 0 & 0.0259\end{bmatrix}
$$

$$
\frac{\partial L}{\partial b^{[1]}} = \begin{bmatrix}0.0466 \\ 0.0372 \\ 0.0259\end{bmatrix}
$$

> **Observe:** The middle column of $\frac{\partial L}{\partial W^{[1]}}$ is all zeros because $x_2 = 0$. Weights connecting to a zero input contribute nothing to the forward pass for this example and receive no gradient update. Correct and expected.


In [ ]:
# ── Example 3: NumPy verification ────────────────────────────────────────

x3 = np.array([[1.0], [0.0], [1.0]])
y3 = np.array([[1.0], [0.0]])

W1c = np.array([[0.10, 0.20, 0.30],
                [0.40, 0.50, 0.60],
                [0.70, 0.80, 0.90]])
b1c = np.array([[0.10], [0.10], [0.10]])
W2c = np.array([[0.20, 0.30, 0.40],
                [0.50, 0.60, 0.70]])
b2c = np.array([[0.10], [0.10]])

# Forward
z1c   = W1c @ x3 + b1c
a1c   = sigmoid(z1c)
z2c   = W2c @ a1c + b2c
yhatc = softmax(z2c)
Lc    = cross_entropy(yhatc, y3)

print("─── Example 3: Forward Pass ───")
print(f"z1      = {z1c.ravel()}")
print(f"a1      = {a1c.ravel()}")
print(f"z2      = {z2c.ravel()}")
print(f"y_hat   = {yhatc.ravel()}")
print(f"Loss    = {Lc:.6f}")

# Backward
delta2c  = yhatc - y3
dW2c     = delta2c @ a1c.T
db2c     = delta2c
dL_da1c  = W2c.T @ delta2c
delta1c  = dL_da1c * sigmoid_d(a1c)    # ⊙ σ'(z) computed from a
dW1c     = delta1c @ x3.T
db1c     = delta1c

print("\n─── Example 3: Backward Pass ───")
print(f"delta2  = {delta2c.ravel()}")
print(f"dL/da1  = {dL_da1c.ravel()}  ← coincidentally all equal (see note above)")
print(f"delta1  = {delta1c.ravel()}")
print(f"dW1     =\n{dW1c}")
print(f"dW2     =\n{dW2c}")


─── Example 3: Forward Pass ───
z1      = [0.5 1.1 1.7]
a1      = [0.622459 0.75026  0.845535]
z2      = [0.787784 1.45326 ]
y_hat   = [0.339511 0.660489]
Loss    = 1.080250

─── Example 3: Backward Pass ───
delta2  = [-0.660489  0.660489]
dL/da1  = [0.198147 0.198147 0.198147]  ← coincidentally all equal (see note above)
delta1  = [0.046565 0.037127 0.025879]
dW1     =
[[0.046565 0.       0.046565]
 [0.037127 0.       0.037127]
 [0.025879 0.       0.025879]]
dW2     =
[[-0.411128 -0.495539 -0.558467]
 [ 0.411128  0.495539  0.558467]]


---
## Part 7 — Numerical Gradient Checking

### What is it?

Before trusting any backprop implementation, verify it against **finite differences**. The idea: numerically approximate $\frac{\partial L}{\partial w}$ by perturbing $w$ slightly and measuring the change in $L$.

$$
\frac{\partial L}{\partial w} \approx \frac{L(w + \varepsilon) - L(w - \varepsilon)}{2\varepsilon}
$$

This is the **centred difference formula**. It has error $O(\varepsilon^2)$, much smaller than the one-sided formula's $O(\varepsilon)$.

### How close is close enough?

Compute the **relative error**:

$$
\text{relative error} = \frac{|\text{analytic gradient} - \text{numerical gradient}|}{|\text{analytic gradient}| + |\text{numerical gradient}|}
$$

| Relative error | Interpretation |
|----------------|---------------|
| $< 10^{-5}$ | Very good — backprop is almost certainly correct |
| $10^{-4}$ to $10^{-3}$ | Acceptable — small numerical errors are expected |
| $> 10^{-2}$ | Bug in backprop — investigate |

### When to use it

- Whenever you implement a new layer, activation, or loss
- After any refactoring of the backward pass
- As a final sanity check before training

> **Exam context:** You may be asked to explain gradient checking or compute a finite-difference approximation for a single weight.


In [ ]:
# ── Numerical gradient check for Example 1 ───────────────────────────────

def forward_pass_ex1(W1, b1, W2, b2, x, y):
    """Returns scalar loss for Example 1 (sigmoid + MSE)."""
    z1    = W1 @ x + b1
    a1    = sigmoid(z1)
    z2    = W2 @ a1 + b2
    y_hat = sigmoid(z2)
    return mse_loss(y_hat, y)

def numerical_gradient(param, idx, W1, b1, W2, b2, x, y, eps=1e-5):
    """Finite-difference gradient for a single parameter element."""
    param_flat = param.ravel()
    orig = param_flat[idx]

    param_flat[idx] = orig + eps
    L_plus = forward_pass_ex1(W1, b1, W2, b2, x, y)

    param_flat[idx] = orig - eps
    L_minus = forward_pass_ex1(W1, b1, W2, b2, x, y)

    param_flat[idx] = orig          # restore
    return (L_plus - L_minus) / (2 * eps)

# Re-run Example 1 forward + backward to get analytic gradients
x  = np.array([[1.0], [0.0]])
y  = np.array([[1.0]])
W1 = np.array([[0.1, 0.2], [0.3, 0.4]])
b1 = np.array([[0.1], [0.1]])
W2 = np.array([[0.5, 0.6]])
b2 = np.array([[0.1]])

z1    = W1 @ x + b1; a1 = sigmoid(z1)
z2    = W2 @ a1 + b2; y_hat = sigmoid(z2)
delta2 = (y_hat - y) * sigmoid_d(y_hat)
dW2   = delta2 @ a1.T
delta1 = (W2.T @ delta2) * sigmoid_d(a1)
dW1   = delta1 @ x.T

print(f"{'Param':12} {'Analytic':>14} {'Numerical':>14} {'Rel. error':>14}")
print("─" * 58)

checks = [
    ("W2[0,0]", W2, dW2, 0, 0),
    ("W2[0,1]", W2, dW2, 0, 1),
    ("W1[0,0]", W1, dW1, 0, 0),
    ("W1[1,0]", W1, dW1, 1, 0),
    ("W1[0,1]", W1, dW1, 0, 1),   # expect ~0 because x2=0
]

for name, param, grad, r, c in checks:
    analytic = grad[r, c]
    # Pass W1, W2, b1, b2 directly (not copies) so that in-place
    # perturbation of `param` is seen by forward_pass_ex1.
    # numerical_gradient restores the original value after each call.
    numeric  = numerical_gradient(param, r * param.shape[1] + c,
                                  W1, b1, W2, b2, x, y)
    rel_err  = abs(analytic - numeric) / (abs(analytic) + abs(numeric) + 1e-15)
    flag = "✓" if rel_err < 1e-4 else "✗ BUG"
    print(f"{name:12} {analytic:14.8f} {numeric:14.8f} {rel_err:14.2e}  {flag}")


Param              Analytic      Numerical     Rel. error
──────────────────────────────────────────────────────────
W2[0,0]         -0.03907126    -0.03907126       2.93e-11  ✓
W2[0,1]         -0.04254280    -0.04254280       5.19e-13  ✓
W1[0,0]         -0.00879428    -0.00879428       6.34e-11  ✓
W1[1,0]         -0.01024377    -0.01024377       3.72e-11  ✓
W1[0,1]          0.00000000     0.00000000       0.00e+00  ✓


---
## Part 8 — One Full Training Loop

Below we run 200 gradient descent steps on Example 1 and track the loss. This confirms that the computed gradients actually reduce the loss — the ultimate validation.


In [ ]:
# ── Full training loop: Example 1 (Sigmoid + MSE) ────────────────────────

np.random.seed(0)
W1 = np.array([[0.1, 0.2], [0.3, 0.4]])
b1 = np.array([[0.1], [0.1]])
W2 = np.array([[0.5, 0.6]])
b2 = np.array([[0.1]])
x  = np.array([[1.0], [0.0]])
y  = np.array([[1.0]])
lr = 0.5

losses = []

for step in range(200):
    # Forward
    z1    = W1 @ x + b1;  a1 = sigmoid(z1)
    z2    = W2 @ a1 + b2; y_hat = sigmoid(z2)
    L     = mse_loss(y_hat, y)
    losses.append(float(L))

    # Backward
    delta2 = (y_hat - y) * sigmoid_d(y_hat)
    dW2 = delta2 @ a1.T;  db2 = delta2
    delta1 = (W2.T @ delta2) * sigmoid_d(a1)
    dW1 = delta1 @ x.T;  db1 = delta1

    # Update
    W1 -= lr * dW1;  b1 -= lr * db1
    W2 -= lr * dW2;  b2 -= lr * db2

print(f"Initial loss : {losses[0]:.6f}")
print(f"Loss at step 10  : {losses[10]:.6f}")
print(f"Loss at step 50  : {losses[50]:.6f}")
print(f"Loss at step 199 : {losses[-1]:.6f}")
print(f"\nFinal y_hat  : {y_hat.ravel()}  (target: {y.ravel()})")
print(f"\n✓ Loss decreases monotonically: {all(losses[i] >= losses[i+1] for i in range(len(losses)-1))}")


Initial loss : 0.052582
Loss at step 10  : 0.025922
Loss at step 50  : 0.006641
Loss at step 199 : 0.001433

Final y_hat  : [0.946462]  (target: [1.])

✓ Loss decreases monotonically: True


---
## Part 9 — Vanishing Gradients

### Why sigmoid hidden layers are problematic in deep networks

The sigmoid derivative has a hard upper bound:

$$
\sigma'(z) = \sigma(z)(1-\sigma(z)) \leq 0.25
$$

This maximum is achieved only at $z = 0$ (where $\sigma(0) = 0.5$). As $|z|$ grows, $\sigma'(z) \to 0$.

### How gradients shrink through layers

Each time a gradient passes through a sigmoid layer, it is multiplied by $\sigma'(z) \leq 0.25$. For a network with $L$ hidden layers:

$$
\delta^{[1]} \approx \delta^{[L]} \times w_1 \times \sigma'(z^{[L-1]}) \times w_2 \times \sigma'(z^{[L-2]}) \times \cdots
$$

After $L$ layers, the gradient has been multiplied by at most $0.25^L$:

| Layers | Max gradient factor |
|--------|---------------------|
| 1 | $0.25$ |
| 4 | $0.25^4 \approx 0.004$ |
| 8 | $0.25^8 \approx 1.5 \times 10^{-5}$ |

Early layers receive near-zero gradients and barely learn — this is the **vanishing gradient problem**.

### Why ReLU helps

ReLU's derivative is either 0 or 1. When it is 1, the gradient passes through completely unchanged (no shrinkage). This is the primary reason ReLU replaced sigmoid in hidden layers for deep networks.

### Exam-ready summary

| Activation | Derivative range | Vanishing gradient risk |
|------------|-----------------|------------------------|
| Sigmoid | $(0, 0.25]$ | High — each layer shrinks gradient by ≥ 4× |
| Tanh | $(0, 1]$ | Moderate — better than sigmoid, still < 1 |
| ReLU | $\{0, 1\}$ | Low — gradient passes through when active |


In [ ]:
# ── Visualise vanishing gradients ─────────────────────────────────────────

z_range = np.linspace(-5, 5, 400)
sig_vals = sigmoid(z_range)
sig_d_vals = sigmoid_d(sig_vals)

print("Sigmoid derivative at key points:")
for z in [-3, -1, 0, 1, 3]:
    a = sigmoid(float(z))
    d = sigmoid_d(a)
    print(f"  z={z:+d} → σ(z)={a:.4f}, σ'(z)={d:.4f}")

print(f"\nMaximum σ'(z) = {sig_d_vals.max():.4f}  (at z=0)")
print(f"σ'(z) at z=±2: {sigmoid_d(sigmoid(2.0)):.4f}")
print(f"\nGradient after 4 sigmoid layers (worst case): {0.25**4:.6f}")
print(f"Gradient after 8 sigmoid layers (worst case): {0.25**8:.2e}")


Sigmoid derivative at key points:
  z=-3 → σ(z)=0.0474, σ'(z)=0.0452
  z=-1 → σ(z)=0.2689, σ'(z)=0.1966
  z=+0 → σ(z)=0.5000, σ'(z)=0.2500
  z=+1 → σ(z)=0.7311, σ'(z)=0.1966
  z=+3 → σ(z)=0.9526, σ'(z)=0.0452

Maximum σ'(z) = 0.2500  (at z=0)
σ'(z) at z=±2: 0.1050

Gradient after 4 sigmoid layers (worst case): 0.003906
Gradient after 8 sigmoid layers (worst case): 1.53e-05


---
## Part 10 — Top 5 Exam Pitfalls in Backpropagation

These are the most common errors in exams and implementations. Read each carefully.

---

### Pitfall 1: Wrong matrix transpose

The gradient formula is:

$$
\frac{\partial L}{\partial W^{[\ell]}} = \delta^{[\ell]} \left(a^{[\ell-1]}\right)^T
$$

**The transpose goes on $a$, not on $\delta$.** A common error is writing $(\delta^{[\ell]})^T a^{[\ell-1]}$, which produces the wrong shape.

Similarly, when propagating the gradient backward:

$$
\frac{\partial L}{\partial a^{[\ell-1]}} = \left(W^{[\ell]}\right)^T \delta^{[\ell]}
$$

**The transpose goes on $W$.** Using $W \delta^T$ or $(W\delta)^T$ are both wrong.

**Dimension check (always do this):**
- $\delta^{[\ell]}$ has shape $(n^{[\ell]}, 1)$
- $a^{[\ell-1]}$ has shape $(n^{[\ell-1]}, 1)$
- $\frac{\partial L}{\partial W^{[\ell]}}$ must have shape $(n^{[\ell]}, n^{[\ell-1]})$ — same as $W^{[\ell]}$
- $\delta^{[\ell]}(a^{[\ell-1]})^T$ gives $(n^{[\ell]}, 1)(1, n^{[\ell-1]}) = (n^{[\ell]}, n^{[\ell-1]})$ ✓

---

### Pitfall 2: Hadamard product vs. matrix multiplication

$$
\delta^{[\ell]} = \underbrace{(W^{[\ell+1]})^T \delta^{[\ell+1]}}_{\text{matrix multiply}} \odot \underbrace{g'(z^{[\ell]})}_{\text{element-wise}}
$$

The gradient from the next layer uses **matrix multiplication** ($(W^{[\ell+1]})^T \delta^{[\ell+1]}$).
The activation derivative uses **element-wise multiplication** ($\odot$).

Using element-wise multiplication for both, or matrix multiplication for both, produces the wrong result and the wrong shape.

---

### Pitfall 3: Sigmoid derivative uses $a$, ReLU derivative uses $z$

| Activation | Derivative argument | Formula |
|------------|---------------------|---------|
| Sigmoid | $a = \sigma(z)$ | $a(1-a)$ |
| ReLU | $z$ (pre-activation) | $\mathbf{1}[z > 0]$ |

For sigmoid: `sigmoid_d(a)` — use the activation already computed in the forward pass.
For ReLU: `relu_d(z)` — must use the pre-activation $z$, not the activation $a$.

Swapping these arguments produces subtly wrong gradients that may not fail loudly.

---

### Pitfall 4: Applying $\delta = \hat{y} - y$ without verifying conditions

The shortcut $\delta^{[L]} = \hat{y} - y$ is **only valid** when:
1. The output activation is **softmax**
2. The loss is **cross-entropy** $L = -\sum_i y_i \log \hat{y}_i$
3. The labels $y$ are **one-hot**

Using this shortcut with sigmoid + MSE, or with multi-label targets, produces wrong gradients.

For sigmoid + MSE: $\delta^{[L]} = (\hat{y} - y) \odot \hat{y}(1-\hat{y})$

---

### Pitfall 5: Forgetting that zero-input weights have zero gradients

If input $x_j = 0$:

$$
\frac{\partial L}{\partial w_{ij}} = \delta_i \cdot x_j = \delta_i \cdot 0 = 0
$$

This is mathematically correct. Zero-input weights are not updated by gradient descent on this example. This is not a bug in your computation. In a batch setting, such weights will receive gradients from other samples where $x_j \neq 0$.


---
## Part 11 — Exercises

**Instructions:** Attempt each exercise by hand first. Compute every intermediate value manually. Then run the solution cell to check your work. Record your numerical answers before revealing the solution.

---

### Exercise 1: 1-2-1 Network, Sigmoid + MSE

A minimal network. No matrix notation — compute each weight scalar by hand.

**Architecture:** 1 input, 2 hidden (sigmoid), 1 output (sigmoid). MSE loss.

**Parameters:**

$$
x = 0.5, \quad y = 1.0
$$

$$
w_{11}^{[1]} = 0.4,\; w_{21}^{[1]} = -0.3,\; b_1^{[1]} = 0.0,\; b_2^{[1]} = 0.0
$$

$$
w_1^{[2]} = 0.7,\; w_2^{[2]} = 0.5,\; b^{[2]} = 0.0
$$

**Tasks:**
1. Compute $z_1^{[1]}, z_2^{[1]}, a_1^{[1]}, a_2^{[1]}$
2. Compute $z^{[2]}, \hat{y}, L$
3. Compute $\delta^{[2]}, \delta_1^{[1]}, \delta_2^{[1]}$
4. Compute all 6 weight gradients and 3 bias gradients

**Hint:** Work exactly like Example 1, substituting scalars.


In [ ]:
# ── Exercise 1 — Solution ─────────────────────────────────────────────────
# Attempt the exercise by hand before running this cell.

x_ex1  = np.array([[0.5]])
y_ex1  = np.array([[1.0]])
W1_ex1 = np.array([[0.4], [-0.3]])         # shape (2,1)
b1_ex1 = np.array([[0.0], [0.0]])
W2_ex1 = np.array([[0.7, 0.5]])            # shape (1,2)
b2_ex1 = np.array([[0.0]])

# Forward
z1_ex1   = W1_ex1 @ x_ex1 + b1_ex1
a1_ex1   = sigmoid(z1_ex1)
z2_ex1   = W2_ex1 @ a1_ex1 + b2_ex1
yhat_ex1 = sigmoid(z2_ex1)
L_ex1    = mse_loss(yhat_ex1, y_ex1)

print("─── Exercise 1 Solution ───")
print(f"z1     = {z1_ex1.ravel()}")
print(f"a1     = {a1_ex1.ravel()}")
print(f"z2     = {z2_ex1.ravel()}")
print(f"y_hat  = {yhat_ex1.ravel()}")
print(f"Loss   = {L_ex1:.6f}")

# Backward
d2_ex1   = (yhat_ex1 - y_ex1) * sigmoid_d(yhat_ex1)
dW2_ex1  = d2_ex1 @ a1_ex1.T
db2_ex1  = d2_ex1
d1_ex1   = (W2_ex1.T @ d2_ex1) * sigmoid_d(a1_ex1)
dW1_ex1  = d1_ex1 @ x_ex1.T
db1_ex1  = d1_ex1

print(f"\ndelta2 = {d2_ex1.ravel()}")
print(f"dW2    = {dW2_ex1.ravel()}  (grad for w_1^[2], w_2^[2])")
print(f"db2    = {db2_ex1.ravel()}")
print(f"delta1 = {d1_ex1.ravel()}")
print(f"dW1    = {dW1_ex1.ravel()}  (grad for w_11^[1], w_21^[1])")
print(f"db1    = {db1_ex1.ravel()}")


─── Exercise 1 Solution ───
z1     = [ 0.2  -0.15]
a1     = [0.549834 0.46257 ]
z2     = [0.616169]
y_hat  = [0.649347]
Loss   = 0.061479

delta2 = [-0.079842]
dW2    = [-0.0439   -0.036933]  (grad for w_1^[2], w_2^[2])
db2    = [-0.079842]
delta1 = [-0.013834 -0.009924]
dW1    = [-0.006917 -0.004962]  (grad for w_11^[1], w_21^[1])
db1    = [-0.013834 -0.009924]


---

### Exercise 2: Dying ReLU Scenario

This exercise tests understanding of the ReLU derivative.

**Architecture:** 1 input, 2 hidden (ReLU), 1 output (sigmoid). MSE loss.

**Parameters:**

$$
x = 2.0, \quad y = 0.0
$$

$$
W^{[1]} = \begin{bmatrix}0.3 \\ -0.5\end{bmatrix}, \quad b^{[1]} = \begin{bmatrix}-1.0 \\ 0.0\end{bmatrix}
$$

$$
W^{[2]} = \begin{bmatrix}0.4 & 0.2\end{bmatrix}, \quad b^{[2]} = 0.1
$$

**Tasks:**
1. Compute $z_1^{[1]}, z_2^{[1]}, a_1^{[1]}, a_2^{[1]}$
2. What is the ReLU derivative for each hidden unit?
3. Complete the full backward pass
4. Which weight receives zero gradient, and why?

**Key question:** After this update, if we set $x = 2.0$ again, will the gradient situation change?


In [ ]:
# ── Exercise 2 — Solution ─────────────────────────────────────────────────

x_ex2  = np.array([[2.0]])
y_ex2  = np.array([[0.0]])
W1_ex2 = np.array([[0.3], [-0.5]])
b1_ex2 = np.array([[-1.0], [0.0]])
W2_ex2 = np.array([[0.4, 0.2]])
b2_ex2 = np.array([[0.1]])

z1_ex2   = W1_ex2 @ x_ex2 + b1_ex2
a1_ex2   = relu(z1_ex2)
z2_ex2   = W2_ex2 @ a1_ex2 + b2_ex2
yhat_ex2 = sigmoid(z2_ex2)
L_ex2    = mse_loss(yhat_ex2, y_ex2)

print("─── Exercise 2 Solution ───")
print(f"z1     = {z1_ex2.ravel()}")
print(f"a1     = {a1_ex2.ravel()}  ← one unit is dead (a=0)!")
print(f"ReLU'  = {relu_d(z1_ex2).ravel()}  ← gradient blocked for dead unit")
print(f"z2     = {z2_ex2.ravel()}")
print(f"y_hat  = {yhat_ex2.ravel()}")
print(f"Loss   = {L_ex2:.6f}")

d2_ex2   = (yhat_ex2 - y_ex2) * sigmoid_d(yhat_ex2)
dW2_ex2  = d2_ex2 @ a1_ex2.T
db2_ex2  = d2_ex2
d1_ex2   = (W2_ex2.T @ d2_ex2) * relu_d(z1_ex2)
dW1_ex2  = d1_ex2 @ x_ex2.T
db1_ex2  = d1_ex2

print(f"\ndelta2 = {d2_ex2.ravel()}")
print(f"delta1 = {d1_ex2.ravel()}  ← delta for dead unit is 0")
print(f"dW1    = {dW1_ex2.ravel()}  ← weight to dead unit gets zero gradient")
print(f"dW2    = {dW2_ex2.ravel()}")
print(f"\nAnswer: Unit 1 (z=-0.4 < 0) is dead. Its weights receive no gradient.")
print(f"After one update, b1[0] does not change. Unit 1 remains dead for x=2.")


─── Exercise 2 Solution ───
z1     = [-0.4 -1. ]
a1     = [0. 0.]  ← one unit is dead (a=0)!
ReLU'  = [0. 0.]  ← gradient blocked for dead unit
z2     = [0.1]
y_hat  = [0.524979]
Loss   = 0.137802

delta2 = [0.130917]
delta1 = [0. 0.]  ← delta for dead unit is 0
dW1    = [0. 0.]  ← weight to dead unit gets zero gradient
dW2    = [0. 0.]

Answer: Unit 1 (z=-0.4 < 0) is dead. Its weights receive no gradient.
After one update, b1[0] does not change. Unit 1 remains dead for x=2.


---

### Exercise 3: Softmax + CE — Identify and fix the error

The following backward pass implementation has a **deliberate bug**. Identify it and write the correct version.

```python
# BUGGY backward pass for softmax + CE (3 output classes)

delta2_bug = y_hat - y           # output delta (this line is correct)

dW2_bug = a1.T @ delta2_bug      # ← is this right?
db2_bug = delta2_bug

dL_da1_bug = W2 @ delta2_bug     # ← is this right?
delta1_bug  = dL_da1_bug * relu_d(z1)
```

**Tasks:**
1. Identify both bugs and explain what each error produces
2. Write the correct two lines
3. Check that dimensions work out for a (3-class, 2-hidden) network

**No computation needed — this is a conceptual exercise.**


In [ ]:
# ── Exercise 3 — Solution ─────────────────────────────────────────────────

print("─── Exercise 3 Solution ───")
print()
print("Bug 1:  dW2_bug = a1.T @ delta2_bug")
print("  WRONG: this transposes a1 and puts it on the left.")
print("  a1 has shape (2,1), a1.T has shape (1,2).")
print("  a1.T @ delta2 → (1,2) @ (3,1) → dimension mismatch.")
print()
print("CORRECT: dW2 = delta2 @ a1.T")
print("  delta2: (3,1), a1.T: (1,2) → result: (3,2) = same shape as W2 ✓")
print()
print("Bug 2:  dL_da1_bug = W2 @ delta2_bug")
print("  WRONG: gradient propagates backward through W2 TRANSPOSED.")
print("  W2 has shape (3,2). W2 @ delta2 → (3,2)@(3,1) → dimension mismatch.")
print()
print("CORRECT: dL_da1 = W2.T @ delta2")
print("  W2.T: (2,3), delta2: (3,1) → result: (2,1) = same shape as a1 ✓")
print()
print("Dimension summary for a 2-hidden, 3-output network:")
print("  W2        : (3,2)")
print("  delta2    : (3,1)")
print("  a1        : (2,1)")
print("  dW2 = delta2 @ a1.T        : (3,1)@(1,2) = (3,2) ✓")
print("  dL_da1 = W2.T @ delta2     : (2,3)@(3,1) = (2,1) ✓")


─── Exercise 3 Solution ───

Bug 1:  dW2_bug = a1.T @ delta2_bug
  WRONG: this transposes a1 and puts it on the left.
  a1 has shape (2,1), a1.T has shape (1,2).
  a1.T @ delta2 → (1,2) @ (3,1) → dimension mismatch.

CORRECT: dW2 = delta2 @ a1.T
  delta2: (3,1), a1.T: (1,2) → result: (3,2) = same shape as W2 ✓

Bug 2:  dL_da1_bug = W2 @ delta2_bug
  WRONG: gradient propagates backward through W2 TRANSPOSED.
  W2 has shape (3,2). W2 @ delta2 → (3,2)@(3,1) → dimension mismatch.

CORRECT: dL_da1 = W2.T @ delta2
  W2.T: (2,3), delta2: (3,1) → result: (2,1) = same shape as a1 ✓

Dimension summary for a 2-hidden, 3-output network:
  W2        : (3,2)
  delta2    : (3,1)
  a1        : (2,1)
  dW2 = delta2 @ a1.T        : (3,1)@(1,2) = (3,2) ✓
  dL_da1 = W2.T @ delta2     : (2,3)@(3,1) = (2,1) ✓


---
## Part 12 — Exam Quick Reference

Write everything below from memory before your exam.

---

### Forward pass (any layer $\ell$)

$$
z^{[\ell]} = W^{[\ell]} a^{[\ell-1]} + b^{[\ell]}, \qquad a^{[\ell]} = g^{[\ell]}(z^{[\ell]})
$$

---

### Backward pass

| Equation | Formula |
|----------|---------|
| Output delta (general) | $\delta^{[L]} = \frac{\partial L}{\partial a^{[L]}} \odot g'^{[L]}(z^{[L]})$ |
| Output delta (softmax + CE) | $\delta^{[L]} = \hat{y} - y$ |
| Hidden delta | $\delta^{[\ell]} = (W^{[\ell+1]})^T \delta^{[\ell+1]} \odot g'^{[\ell]}(z^{[\ell]})$ |
| Weight gradient | $\frac{\partial L}{\partial W^{[\ell]}} = \delta^{[\ell]} (a^{[\ell-1]})^T$ |
| Bias gradient | $\frac{\partial L}{\partial b^{[\ell]}} = \delta^{[\ell]}$ |
| Parameter update | $\theta \leftarrow \theta - \eta \frac{\partial L}{\partial \theta}$ |

---

### Activation derivatives

| Activation | Derivative |
|------------|-----------|
| Sigmoid: $\sigma(z) = \frac{1}{1+e^{-z}}$ | $\sigma'(z) = a(1-a)$, $a = \sigma(z)$ |
| ReLU: $\max(0,z)$ | $\mathbf{1}[z > 0]$ — apply to $z$, not $a$ |
| Softmax + CE combined | $\delta = \hat{y} - y$ |

---

### Loss functions

| Loss | Formula | $\partial L / \partial \hat{y}$ |
|------|---------|-------------------------------|
| MSE | $\frac{1}{2}\sum(\hat{y}-y)^2$ | $\hat{y} - y$ |
| Cross-entropy | $-\sum y_i \log \hat{y}_i$ | $-y_i / \hat{y}_i$ |

---

### Numerical gradient check

$$
\frac{\partial L}{\partial w} \approx \frac{L(w+\varepsilon) - L(w-\varepsilon)}{2\varepsilon}, \qquad \varepsilon = 10^{-5}
$$

Relative error $< 10^{-5}$: correct. $> 10^{-2}$: bug.

---

### Common dimensions (2-hidden, 3-output example)

| Variable | Shape | Notes |
|----------|-------|-------|
| $x, a^{[0]}$ | $(n^{[0]}, 1)$ | input |
| $W^{[\ell]}$ | $(n^{[\ell]}, n^{[\ell-1]})$ | rows = neurons at $\ell$, cols = neurons at $\ell-1$ |
| $b^{[\ell]}, \delta^{[\ell]}, a^{[\ell]}$ | $(n^{[\ell]}, 1)$ | column vectors |
| $\frac{\partial L}{\partial W^{[\ell]}}$ | same as $W^{[\ell]}$ | always |

---

### 5 things to check under exam pressure

1. Did I transpose $a$, not $\delta$, in the weight gradient?
2. Did I use $(W^{[\ell+1]})^T$ (transposed) when backpropagating deltas?
3. Did I use $\odot$ (element-wise) for the activation derivative, not matrix multiply?
4. For sigmoid, am I computing $\sigma'$ from $a$? For ReLU, from $z$?
5. Is $\delta = \hat{y} - y$ valid here (softmax + CE + one-hot)?


---
## Part 13 — Mini-Batch / Vectorised Backpropagation

### Why this matters

Every example so far has used a **single sample**: $x$ is a column vector of shape $(n^{[0]}, 1)$.
In practice — and in exam questions involving batches — we process $m$ samples simultaneously.
The equations change shape, but the logic is identical. Understanding the difference is exam-critical.

---

### Notation change: from vectors to matrices

| Symbol | Single sample $(m=1)$ | Mini-batch $(m>1)$ |
|--------|-----------------------|---------------------|
| Input | $x$ — shape $(n^{[0]}, 1)$ | $X$ — shape $(n^{[0]}, m)$, each column is one sample |
| Activations | $a^{[\ell]}$ — shape $(n^{[\ell]}, 1)$ | $A^{[\ell]}$ — shape $(n^{[\ell]}, m)$ |
| Deltas | $\delta^{[\ell]}$ — shape $(n^{[\ell]}, 1)$ | $\Delta^{[\ell]}$ — shape $(n^{[\ell]}, m)$ |
| Loss | scalar $L$ | average $L = \frac{1}{m}\sum_{i=1}^m L^{(i)}$ |

Weights $W^{[\ell]}$ and biases $b^{[\ell]}$ do **not** change shape — the same parameters serve all $m$ samples.

---

### Forward pass (vectorised)

$$
Z^{[\ell]} = W^{[\ell]} A^{[\ell-1]} + b^{[\ell]}
$$

NumPy broadcasts $b^{[\ell]}$ of shape $(n^{[\ell]}, 1)$ across all $m$ columns automatically.

$$
A^{[\ell]} = g^{[\ell]}(Z^{[\ell]})  \quad \text{(element-wise)}
$$

---

### Backward pass (vectorised)

The only structural change is in the **weight gradient**, which now averages over the batch:

$$
\frac{\partial L}{\partial W^{[\ell]}} = \frac{1}{m} \Delta^{[\ell]} (A^{[\ell-1]})^T
$$

$$
\frac{\partial L}{\partial b^{[\ell]}} = \frac{1}{m} \sum_{i=1}^{m} \delta^{[\ell](i)} = \frac{1}{m} \Delta^{[\ell]} \mathbf{1}
$$

In NumPy: `db = (1/m) * delta.sum(axis=1, keepdims=True)`

The hidden delta is unchanged in structure:

$$
\Delta^{[\ell]} = (W^{[\ell+1]})^T \Delta^{[\ell+1]} \odot g'^{[\ell]}(Z^{[\ell]})
$$

where $\odot$ is still element-wise (broadcast across columns).

---

### Why average, not sum?

If we summed gradients over the batch, a batch of $m=64$ would produce gradients 64× larger than $m=1$, forcing us to scale the learning rate accordingly.
Averaging makes the gradient estimate independent of batch size, so the learning rate $\eta$ remains interpretable.

> **Exam trap:** Some formulations divide by $m$, some do not. In MSE, the factor of $\frac{1}{2}$ and the $\frac{1}{m}$ are both sometimes absorbed. Always check whether the loss definition already includes $\frac{1}{m}$ before deciding whether to divide again in the gradient.


In [ ]:
# ── Part 13: Mini-batch backprop — Example 1 architecture, m=4 samples ──

np.random.seed(42)

# ── Parameters (same architecture as Example 1) ───────────────────────────
W1 = np.array([[0.1, 0.2], [0.3, 0.4]])
b1 = np.array([[0.1], [0.1]])
W2 = np.array([[0.5, 0.6]])
b2 = np.array([[0.1]])

# ── Mini-batch: 4 samples, each with 2 features ───────────────────────────
# X shape: (n_input=2, m=4) — each column is one sample
X = np.array([[1.0, 0.0, 1.0, 0.5],   # x1 values for 4 samples
              [0.0, 1.0, 1.0, 0.5]])   # x2 values for 4 samples

# Y shape: (n_output=1, m=4) — each column is one target
Y = np.array([[1.0, 0.0, 1.0, 0.5]])

m = X.shape[1]   # number of samples = 4

# ── Vectorised forward pass ───────────────────────────────────────────────
Z1 = W1 @ X + b1         # (2,2) @ (2,4) + (2,1) = (2,4)   broadcast b1
A1 = sigmoid(Z1)          # (2,4) element-wise
Z2 = W2 @ A1 + b2         # (1,2) @ (2,4) + (1,1) = (1,4)
Y_hat = sigmoid(Z2)        # (1,4)

# Average MSE over the batch (the 1/m is here, not in the backward pass)
L_batch = (1/m) * mse_loss(Y_hat, Y)   # Note: mse already has ½

print("─── Mini-batch forward pass (m=4) ───")
print(f"X shape     : {X.shape}")
print(f"Z1 shape    : {Z1.shape}  ← one column per sample")
print(f"A1 shape    : {A1.shape}")
print(f"Y_hat shape : {Y_hat.shape}")
print(f"Y_hat       :\n{Y_hat}")
print(f"Y (targets) :\n{Y}")
print(f"Batch loss  : {L_batch:.6f}")


─── Mini-batch forward pass (m=4) ───
X shape     : (2, 4)
Z1 shape    : (2, 4)  ← one column per sample
A1 shape    : (2, 4)
Y_hat shape : (1, 4)
Y_hat       :
[[0.675711 0.681505 0.692818 0.678628]]
Y (targets) :
[[1.  0.  1.  0.5]]
Batch loss  : 0.086985


In [ ]:
# ── Vectorised backward pass ──────────────────────────────────────────────
# Output delta: same formula, applied column-wise
Delta2 = (Y_hat - Y) * sigmoid_d(Y_hat)   # (1,4) — one delta per sample

# Weight & bias gradients: AVERAGE over m samples
dW2 = (1/m) * (Delta2 @ A1.T)    # (1,4)@(4,2) / m = (1,2)
db2 = (1/m) * Delta2.sum(axis=1, keepdims=True)   # (1,1)

# Propagate delta back to hidden layer
Delta1 = (W2.T @ Delta2) * sigmoid_d(A1)   # (2,1)@(1,4) * (2,4) = (2,4)

dW1 = (1/m) * (Delta1 @ X.T)     # (2,4)@(4,2) / m = (2,2)
db1 = (1/m) * Delta1.sum(axis=1, keepdims=True)   # (2,1)

print("─── Mini-batch backward pass (m=4) ───")
print(f"Delta2 shape : {Delta2.shape}  ← one delta column per sample")
print(f"Delta1 shape : {Delta1.shape}")
print(f"dW2 (averaged) : {dW2}")
print(f"db2 (averaged) : {db2}")
print(f"dW1 (averaged) :\n{dW1}")
print(f"db1 (averaged) : {db1.ravel()}")

# ── Verification: single-sample gradient matches column 0 of Delta2 ───────
# For sample 0 alone:
x0, y0 = X[:, [0]], Y[:, [0]]
z1_0 = W1@x0+b1; a1_0 = sigmoid(z1_0)
z2_0 = W2@a1_0+b2; yhat_0 = sigmoid(z2_0)
d2_0 = (yhat_0-y0)*sigmoid_d(yhat_0)
print(f"\nSingle-sample delta for sample 0 : {d2_0.ravel()}")
print(f"Column 0 of batch Delta2          : {Delta2[:, 0]}")
print(f"Match: {np.allclose(d2_0.ravel(), Delta2[:, 0])}")


─── Mini-batch backward pass (m=4) ───
Delta2 shape : (1, 4)  ← one delta column per sample
Delta1 shape : (2, 4)
dW2 (averaged) : [[0.007166 0.007054]]
db2 (averaged) : [[0.012612]]
dW1 (averaged) :
[[-0.003563  0.003156]
 [-0.003964  0.003811]]
db1 (averaged) : [0.001557 0.001945]

Single-sample delta for sample 0 : [-0.07106]
Column 0 of batch Delta2          : [-0.07106]
Match: True


### Exam Checkpoint 3 ✓ — Mini-batch

1. $X$ has shape $(n^{[0]}, m)$; $W^{[\ell]}$ shape is unchanged.
2. Weight gradient is **averaged**: $\frac{1}{m}\Delta^{[\ell]}(A^{[\ell-1]})^T$.
3. Bias gradient is **averaged sum**: $\frac{1}{m}\sum_i \delta_i = \frac{1}{m}\Delta \mathbf{1}$, i.e. `delta.sum(axis=1, keepdims=True) / m`.
4. Element-wise operations ($\odot$, $g'$) broadcast across the $m$ columns automatically.
5. The per-sample delta for sample $i$ is exactly column $i$ of $\Delta^{[\ell]}$.


---
## Part 14 — Tanh Activation

Tanh is a rescaled sigmoid. It appears in RNN cells, some older CNNs, and exam questions that test whether you can adapt backprop to a new activation.

### Definition and derivative

$$
\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}
$$

Key property: $\tanh(z) \in (-1, 1)$, centred at zero (unlike sigmoid, which is in $(0,1)$).

The derivative:

$$
\tanh'(z) = 1 - \tanh^2(z) = 1 - a^2 \qquad \text{where } a = \tanh(z)
$$

Like sigmoid, compute from the activation $a$ — you already have it from the forward pass.

### Comparison with sigmoid

| Property | Sigmoid | Tanh |
|----------|---------|------|
| Output range | $(0, 1)$ | $(-1, 1)$ |
| Zero-centred | No — mean output $\approx 0.5$ | Yes — mean output $\approx 0$ |
| Max derivative | $0.25$ at $z=0$ | $1.0$ at $z=0$ |
| Saturation | Yes at $\pm\infty$ | Yes at $\pm\infty$ |

> **Why zero-centring matters:** If hidden activations are always positive (sigmoid), the gradient $\frac{\partial L}{\partial W}=\delta a^T$ will have all entries of the same sign for a given sample. This constrains weight updates to move together (zig-zag problem). Tanh avoids this.

> **Vanishing gradient comparison:** Tanh's maximum derivative is $1.0$ vs. sigmoid's $0.25$, so tanh vanishes 4× more slowly. It is still not as good as ReLU for deep nets, but significantly better than sigmoid.

---

### Worked Example — 2-1-1 Network, Tanh Hidden + Sigmoid Output + MSE

$$
x_1 = 1.5,\; x_2 = -0.5,\; y = 1
$$

$$
W^{[1]} = \begin{bmatrix}0.3 & -0.2\end{bmatrix},\quad b^{[1]} = 0.1
$$

$$
W^{[2]} = \begin{bmatrix}0.8\end{bmatrix},\quad b^{[2]} = -0.1
$$

**Forward pass:**

$$
z^{[1]} = (0.3)(1.5) + (-0.2)(-0.5) + 0.1 = 0.45 + 0.10 + 0.10 = 0.65
$$

$$
a^{[1]} = \tanh(0.65) \approx 0.5716
$$

$$
z^{[2]} = (0.8)(0.5716) + (-0.1) = 0.4573 - 0.10 = 0.3573
$$

$$
\hat{y} = \sigma(0.3573) \approx 0.5883, \quad L = \frac{1}{2}(0.5883-1)^2 \approx 0.0847
$$

**Backward pass:**

$$
\delta^{[2]} = (\hat{y}-y)\cdot \hat{y}(1-\hat{y}) = (-0.4117)(0.2421) \approx -0.0997
$$

$$
\frac{\partial L}{\partial w^{[2]}} = \delta^{[2]} \cdot a^{[1]} = (-0.0997)(0.5716) \approx -0.0570
$$

$$
\frac{\partial L}{\partial b^{[2]}} = \delta^{[2]} \approx -0.0997
$$

Hidden delta — **tanh derivative** $= 1 - a^2$:

$$
\tanh'(z^{[1]}) = 1 - (0.5716)^2 = 1 - 0.3267 = 0.6733
$$

$$
\delta^{[1]} = \delta^{[2]} \cdot w^{[2]} \cdot \tanh'(z^{[1]}) = (-0.0997)(0.8)(0.6733) \approx -0.0537
$$

$$
\frac{\partial L}{\partial w_{11}^{[1]}} = \delta^{[1]} \cdot x_1 = (-0.0537)(1.5) \approx -0.0805
$$

$$
\frac{\partial L}{\partial w_{12}^{[1]}} = \delta^{[1]} \cdot x_2 = (-0.0537)(-0.5) \approx 0.0268
$$

$$
\frac{\partial L}{\partial b^{[1]}} = \delta^{[1]} \approx -0.0537
$$


In [ ]:
# ── Part 14: Tanh activation — NumPy verification ────────────────────────

def tanh_d(a):
    """Tanh derivative from activation a = tanh(z).
    tanh'(z) = 1 - tanh²(z) = 1 - a²
    Apply to activation a, not to z (same convention as sigmoid_d).
    """
    return 1.0 - a ** 2

# Parameters
x_t  = np.array([[1.5], [-0.5]])
y_t  = np.array([[1.0]])
W1_t = np.array([[0.3, -0.2]])    # shape (1,2)
b1_t = np.array([[0.1]])
W2_t = np.array([[0.8]])           # shape (1,1)
b2_t = np.array([[-0.1]])

# Forward
z1_t    = W1_t @ x_t + b1_t      # (1,1)
a1_t    = np.tanh(z1_t)           # (1,1)
z2_t    = W2_t @ a1_t + b2_t      # (1,1)
yhat_t  = sigmoid(z2_t)           # (1,1)  sigmoid output
L_t     = mse_loss(yhat_t, y_t)

print("─── Tanh example: Forward Pass ───")
print(f"z1     = {z1_t.ravel()}")       # [0.65]
print(f"a1     = {a1_t.ravel()}")       # [0.5716]
print(f"tanh'  = {tanh_d(a1_t).ravel()}")  # [0.6733]
print(f"z2     = {z2_t.ravel()}")       # [0.3573]
print(f"y_hat  = {yhat_t.ravel()}")     # [0.5883]
print(f"Loss   = {L_t:.6f}")            # ~0.0847

# Backward — same structure, only activation derivative changes
d2_t   = (yhat_t - y_t) * sigmoid_d(yhat_t)   # output still sigmoid
dW2_t  = d2_t @ a1_t.T
db2_t  = d2_t

# Hidden: use tanh_d(a), NOT sigmoid_d
d1_t   = (W2_t.T @ d2_t) * tanh_d(a1_t)       # ← tanh derivative
dW1_t  = d1_t @ x_t.T
db1_t  = d1_t

print("\n─── Tanh example: Backward Pass ───")
print(f"delta2 = {d2_t.ravel()}")       # [-0.0997]
print(f"dW2    = {dW2_t.ravel()}")      # [-0.0570]
print(f"db2    = {db2_t.ravel()}")      # [-0.0997]
print(f"delta1 = {d1_t.ravel()}")       # [-0.0537]
print(f"dW1    = {dW1_t.ravel()}")      # [-0.0805,  0.0268]
print(f"db1    = {db1_t.ravel()}")      # [-0.0537]


─── Tanh example: Forward Pass ───
z1     = [0.65]
a1     = [0.57167]
tanh'  = [0.673193]
z2     = [0.357336]
y_hat  = [0.588395]
Loss   = 0.084709

─── Tanh example: Backward Pass ───
delta2 = [-0.099685]
dW2    = [-0.056987]
db2    = [-0.099685]
delta1 = [-0.053686]
dW1    = [-0.080529  0.026843]
db1    = [-0.053686]


### Exam Checkpoint 4 ✓ — Tanh

1. $\tanh'(z) = 1 - a^2$ where $a = \tanh(z)$ — compute from activation, same convention as sigmoid.
2. Maximum $\tanh'$ is $1.0$ (at $z=0$) vs. $0.25$ for sigmoid — tanh is 4× better for gradient flow.
3. The backward pass structure is **identical** to sigmoid — only the local derivative changes.
4. Tanh is zero-centred; sigmoid is not. This affects the sign distribution of gradients.
5. Do not confuse $\tanh'(z)$ with $\sigma'(z)$. Write the formula explicitly on the exam.


---
## Part 15 — L2 Regularisation: How It Modifies Gradients

Regularisation is frequently tested in DLA exams because it modifies the gradient in a specific, predictable way. You must be able to write the regularised gradient from memory.

### The regularised loss

Standard loss plus an L2 penalty on all weights:

$$
L_{\text{reg}} = L + \frac{\lambda}{2m} \sum_{\ell} \|W^{[\ell]}\|_F^2
$$

where $\|\cdot\|_F$ is the Frobenius norm (sum of squares of all entries) and $\lambda$ is the regularisation strength.

The $\frac{1}{2m}$ normalises by batch size $m$ and removes the factor of 2 in the derivative.

### Effect on the gradient

For each weight matrix $W^{[\ell]}$:

$$
\frac{\partial L_{\text{reg}}}{\partial W^{[\ell]}}
= \underbrace{\frac{\partial L}{\partial W^{[\ell]}}}_{\text{original gradient}}
+ \underbrace{\frac{\lambda}{m} W^{[\ell]}}_{\text{regularisation term}}
$$

The update rule becomes:

$$
W^{[\ell]} \leftarrow W^{[\ell]} - \eta \left(\frac{\partial L}{\partial W^{[\ell]}} + \frac{\lambda}{m} W^{[\ell]}\right)
= \underbrace{\left(1 - \frac{\eta\lambda}{m}\right)}_{\text{weight decay}} W^{[\ell]} - \eta \frac{\partial L}{\partial W^{[\ell]}}
$$

This is why L2 regularisation is also called **weight decay** — it multiplies the weights by a factor slightly less than 1 at every step, shrinking them toward zero.

### What regularisation does NOT change

- The delta propagation equations $\delta^{[\ell]}$ are unchanged
- Only the final weight gradient has the extra term added
- Bias gradients are typically **not** regularised (biases are excluded from the penalty)

### When $\lambda$ is large

A large $\lambda$ strongly penalises large weights. The network is forced to distribute its representation across many small weights rather than concentrating it in a few large ones — reducing overfitting.

### Intuition

Without regularisation, a weight can grow arbitrarily large if it reduces training loss. With L2, every large weight is penalised directly, regardless of whether it helps the loss. The network cannot "buy" accuracy with a large weight without paying a regularisation cost.


In [ ]:
# ── Part 15: L2 regularised gradient — Example 1 ─────────────────────────

# Re-initialise Example 1 parameters
W1 = np.array([[0.1, 0.2], [0.3, 0.4]])
b1 = np.array([[0.1], [0.1]])
W2 = np.array([[0.5, 0.6]])
b2 = np.array([[0.1]])
x  = np.array([[1.0], [0.0]])
y  = np.array([[1.0]])

lam = 0.1   # regularisation strength λ
m   = 1     # single sample (m=1)

# Forward
z1    = W1@x+b1; a1 = sigmoid(z1)
z2    = W2@a1+b2; y_hat = sigmoid(z2)
L     = mse_loss(y_hat, y)
L_reg = L + (lam / (2*m)) * (np.sum(W1**2) + np.sum(W2**2))

# Standard backward (unregularised gradients)
d2     = (y_hat-y)*sigmoid_d(y_hat)
dW2    = d2 @ a1.T
db2    = d2
d1     = (W2.T@d2)*sigmoid_d(a1)
dW1    = d1 @ x.T
db1    = d1

# Add regularisation term to weight gradients (NOT to bias gradients)
dW2_reg = dW2 + (lam/m) * W2
dW1_reg = dW1 + (lam/m) * W1

print("─── L2 Regularised Gradients (λ=0.1) ───")
print(f"L (original) : {L:.6f}")
print(f"L (reg)      : {L_reg:.6f}  ← larger due to penalty")
print()
print(f"dW2 (no reg) : {dW2.ravel()}")
print(f"dW2 (reg)    : {dW2_reg.ravel()}")
print(f"  Difference : {(dW2_reg - dW2).ravel()}  ← = (λ/m)·W2 = {(lam/m)*W2.ravel()}")
print()
print(f"dW1 (no reg) :\n{dW1}")
print(f"dW1 (reg)    :\n{dW1_reg}")
print(f"  Difference :\n{dW1_reg - dW1}  ← = (λ/m)·W1")
print()
print(f"db2 (unchanged) : {db2.ravel()}  ← biases not regularised")
print(f"db1 (unchanged) : {db1.ravel()}")

# Weight decay factor
eta = 0.1
decay = 1 - eta*lam/m
print(f"\nWeight decay factor (1 - η·λ/m) = {decay:.4f}")
print(f"Each weight is multiplied by {decay:.4f} before subtracting the gradient.")


─── L2 Regularised Gradients (λ=0.1) ───
L (original) : 0.052582
L (reg)      : 0.098082  ← larger due to penalty

dW2 (no reg) : [-0.039071 -0.042543]
dW2 (reg)    : [0.010929 0.017457]
  Difference : [0.05 0.06]  ← = (λ/m)·W2 = [0.05 0.06]

dW1 (no reg) :
[[-0.008794  0.      ]
 [-0.010244  0.      ]]
dW1 (reg)    :
[[0.001206 0.02    ]
 [0.019756 0.04    ]]
  Difference :
[[0.01 0.02]
 [0.03 0.04]]  ← = (λ/m)·W1

db2 (unchanged) : [-0.07106]  ← biases not regularised
db1 (unchanged) : [-0.008794 -0.010244]

Weight decay factor (1 - η·λ/m) = 0.9900
Each weight is multiplied by 0.9900 before subtracting the gradient.


### Exam Checkpoint 5 ✓ — L2 Regularisation

1. Regularised gradient: $\frac{\partial L_{\text{reg}}}{\partial W^{[\ell]}} = \frac{\partial L}{\partial W^{[\ell]}} + \frac{\lambda}{m} W^{[\ell]}$
2. **Biases are not regularised** — add the $\frac{\lambda}{m} W$ term only to weight gradients, never to bias gradients.
3. The delta equations ($\delta^{[\ell]}$) are unchanged — regularisation only appears at the final gradient step.
4. Weight decay interpretation: the update is equivalent to multiplying each weight by $(1 - \frac{\eta\lambda}{m})$ before the gradient step.
5. Large $\lambda$ → strong shrinkage → underfitting risk. Small $\lambda$ → weak penalty → overfitting risk.


---
## Part 16 — Additional Exercises

---

### Exercise 4: Mini-batch gradient averaging

Two training samples pass through the same 1-1-1 sigmoid network.

$$
W^{[1]} = 0.5,\quad b^{[1]} = 0.0,\quad W^{[2]} = 0.8,\quad b^{[2]} = 0.0
$$

**Sample A:** $x^{(A)} = 1.0,\; y^{(A)} = 1.0$

**Sample B:** $x^{(B)} = -1.0,\; y^{(B)} = 0.0$

**Tasks (by hand):**

1. Compute the forward pass for each sample separately.
2. Compute $\delta^{[2]}$ and $\delta^{[1]}$ for each sample.
3. Compute $\frac{\partial L}{\partial W^{[2]}}$ for each sample.
4. Compute the batch-averaged gradient $\overline{\frac{\partial L}{\partial W^{[2]}}}$.
5. After one gradient descent step with $\eta = 0.5$, what is the new $W^{[2]}$?

**Key question:** Does the averaged gradient point in the same direction as either individual gradient?


In [ ]:
# ── Exercise 4 — Solution ─────────────────────────────────────────────────
# Attempt by hand first.

# Scalar wrapper: work with 1×1 matrices so shapes are explicit
W1_e4 = np.array([[0.5]]); b1_e4 = np.array([[0.0]])
W2_e4 = np.array([[0.8]]); b2_e4 = np.array([[0.0]])

def fwd_e4(x, W1, b1, W2, b2):
    z1 = W1*x + b1; a1 = sigmoid(z1)
    z2 = W2*a1 + b2; yh = sigmoid(z2)
    return z1, a1, z2, yh

def bwd_e4(x, y, a1, yh, W2):
    d2 = (yh-y)*sigmoid_d(yh)
    dW2 = d2*a1
    d1  = (W2*d2)*sigmoid_d(a1)
    dW1 = d1*x
    return d2, dW2, d1, dW1

xA, yA = 1.0, 1.0
xB, yB = -1.0, 0.0

z1A, a1A, z2A, yhA = fwd_e4(xA, W1_e4, b1_e4, W2_e4, b2_e4)
z1B, a1B, z2B, yhB = fwd_e4(xB, W1_e4, b1_e4, W2_e4, b2_e4)

d2A, dW2A, d1A, dW1A = bwd_e4(xA, yA, a1A, yhA, W2_e4)
d2B, dW2B, d1B, dW1B = bwd_e4(xB, yB, a1B, yhB, W2_e4)

dW2_avg = 0.5*(dW2A + dW2B)
eta = 0.5
W2_new = W2_e4 - eta*dW2_avg

print("─── Exercise 4 Solution ───")
print(f"\nSample A: yhat={yhA.ravel()[0]:.4f},  dW2={(dW2A.ravel()[0]):.6f}")
print(f"Sample B: yhat={yhB.ravel()[0]:.4f},  dW2={(dW2B.ravel()[0]):.6f}")
print(f"\ndW2_avg = ({dW2A.ravel()[0]:.6f} + {dW2B.ravel()[0]:.6f}) / 2 = {dW2_avg.ravel()[0]:.6f}")
print(f"W2_new  = {W2_e4.ravel()[0]:.4f} - {eta}×{dW2_avg.ravel()[0]:.6f} = {W2_new.ravel()[0]:.6f}")
print(f"\nSample A gradient is {'negative' if dW2A<0 else 'positive'} (wants W2 to {'increase' if dW2A<0 else 'decrease'})")
print(f"Sample B gradient is {'negative' if dW2B<0 else 'positive'} (wants W2 to {'increase' if dW2B<0 else 'decrease'})")
print(f"Average gradient is {'negative' if dW2_avg<0 else 'positive'} — a compromise between conflicting signals.")


─── Exercise 4 Solution ───

Sample A: yhat=0.6220,  dW2=-0.055324
Sample B: yhat=0.5749,  dW2=0.053047

dW2_avg = (-0.055324 + 0.053047) / 2 = -0.001139
W2_new  = 0.8000 - 0.5×-0.001139 = 0.800569

Sample A gradient is negative (wants W2 to increase)
Sample B gradient is positive (wants W2 to decrease)
Average gradient is negative — a compromise between conflicting signals.


---

### Exercise 5: L2-regularised update with weight decay

Use the same network as Example 1 (2-2-1, sigmoid + MSE), with:

$$
\eta = 0.1,\quad \lambda = 0.5,\quad m = 1
$$

**Tasks:**

1. Using the gradients already computed in Part 4 (Example 1), write the regularised gradient for $W^{[2]}$.
2. What is the weight decay factor $(1 - \frac{\eta\lambda}{m})$ for these hyperparameters?
3. Compute the updated $W^{[2]}$ using the regularised gradient.
4. How different is this from the unregularised update in Part 4?
5. If $\lambda$ were increased to $5.0$, would the gradient be dominated by the regularisation term or the loss term? Calculate.


In [ ]:
# ── Exercise 5 — Solution ─────────────────────────────────────────────────

# Recall from Part 4 (Example 1):
W2_orig  = np.array([[0.5, 0.6]])
dW2_ex1  = np.array([[-0.03907126, -0.04254280]])  # from Example 1 backward pass
eta      = 0.1
lam      = 0.5
m        = 1

print("─── Exercise 5 Solution ───")

# 1. Regularised gradient
dW2_reg = dW2_ex1 + (lam/m)*W2_orig
print(f"\n1. Regularised dW2:")
print(f"   Original dW2    : {dW2_ex1.ravel()}")
print(f"   Reg term (λ/m)·W: {((lam/m)*W2_orig).ravel()}")
print(f"   Total           : {dW2_reg.ravel()}")

# 2. Weight decay factor
decay = 1 - eta*lam/m
print(f"\n2. Weight decay factor: 1 - η·λ/m = 1 - {eta}·{lam}/{m} = {decay:.4f}")
print(f"   Each W2 entry is scaled by {decay:.4f} before the gradient step.")

# 3. Updated W2
W2_unreg = W2_orig - eta*dW2_ex1
W2_reg   = W2_orig - eta*dW2_reg
print(f"\n3. Updated W2:")
print(f"   No regularisation : {W2_unreg.ravel()}")
print(f"   With λ=0.5        : {W2_reg.ravel()}")
print(f"   Difference        : {(W2_reg - W2_unreg).ravel()}")

# 4. Difference explanation
print(f"\n4. The regularised update is smaller (weights are pulled toward 0).")
print(f"   Difference = -η·(λ/m)·W2 = {(-eta*(lam/m)*W2_orig).ravel()}")

# 5. If λ=5.0
lam_large = 5.0
loss_term = np.abs(dW2_ex1)
reg_term  = (lam_large/m)*np.abs(W2_orig)
print(f"\n5. With λ={lam_large}:")
print(f"   |loss gradient|   : {loss_term.ravel()}")
print(f"   |reg term|        : {reg_term.ravel()}")
print(f"   Reg/Loss ratio    : {(reg_term/loss_term).ravel()}")
print(f"   → Dominated by regularisation (reg term is {(reg_term/loss_term).ravel().mean():.0f}× larger).")
print(f"   At λ=5, the network is being shrunk far more than it is being optimised for the loss.")


─── Exercise 5 Solution ───

1. Regularised dW2:
   Original dW2    : [-0.039071 -0.042543]
   Reg term (λ/m)·W: [0.25 0.3 ]
   Total           : [0.210929 0.257457]

2. Weight decay factor: 1 - η·λ/m = 1 - 0.1·0.5/1 = 0.9500
   Each W2 entry is scaled by 0.9500 before the gradient step.

3. Updated W2:
   No regularisation : [0.503907 0.604254]
   With λ=0.5        : [0.478907 0.574254]
   Difference        : [-0.025 -0.03 ]

4. The regularised update is smaller (weights are pulled toward 0).
   Difference = -η·(λ/m)·W2 = [-0.025 -0.03 ]

5. With λ=5.0:
   |loss gradient|   : [0.039071 0.042543]
   |reg term|        : [2.5 3. ]
   Reg/Loss ratio    : [63.985651 70.51722 ]
   → Dominated by regularisation (reg term is 67× larger).
   At λ=5, the network is being shrunk far more than it is being optimised for the loss.


---
## Part 16 Addendum — Expanded Exam Quick Reference

Supplement to Part 12. Add these to your memory cards.

---

### Mini-batch vectorisation

| Equation | Formula |
|----------|---------|
| Weight gradient (batch) | $\frac{1}{m}\Delta^{[\ell]}(A^{[\ell-1]})^T$ |
| Bias gradient (batch) | $\frac{1}{m}\Delta^{[\ell]}\mathbf{1}$ — i.e. `delta.sum(axis=1, keepdims=True)/m` |
| Hidden delta (batch) | $(W^{[\ell+1]})^T\Delta^{[\ell+1]} \odot g'(Z^{[\ell]})$ — same, columns broadcast |

---

### Tanh

| Formula | Value |
|---------|-------|
| $\tanh'(z)$ | $1 - a^2$, $a = \tanh(z)$ |
| Max derivative | $1.0$ at $z=0$ |
| Output range | $(-1, 1)$ |

---

### L2 Regularisation

| Formula | Notes |
|---------|-------|
| Regularised gradient | $\frac{\partial L}{\partial W^{[\ell]}} + \frac{\lambda}{m}W^{[\ell]}$ |
| Biases | Not regularised — no extra term |
| Update (weight decay form) | $(1 - \frac{\eta\lambda}{m})W^{[\ell]} - \eta\frac{\partial L}{\partial W^{[\ell]}}$ |
| Decay factor | $(1 - \frac{\eta\lambda}{m})$ — must be $<1$ for decay to work |

---

### Activation cheat sheet (complete)

| Activation | $g(z)$ | $g'(z)$ | Argument | Max $g'$ |
|------------|--------|---------|----------|---------|
| Sigmoid | $\frac{1}{1+e^{-z}}$ | $a(1-a)$ | from $a$ | $0.25$ |
| Tanh | $\frac{e^z-e^{-z}}{e^z+e^{-z}}$ | $1-a^2$ | from $a$ | $1.0$ |
| ReLU | $\max(0,z)$ | $\mathbf{1}[z>0]$ | from $z$ | $1$ |
| Softmax+CE | — | $\hat{y}-y$ | — | — |

---

### Complete backprop recipe (write this from memory)

Given layer $\ell$ with $W^{[\ell]}, b^{[\ell]}, g^{[\ell]}$:

1. **Store** $Z^{[\ell]}, A^{[\ell]}$ during forward pass — you need them later.
2. **Receive** $\Delta^{[\ell]}$ from the layer above (or compute from loss at output).
3. **Compute** $\frac{\partial L}{\partial W^{[\ell]}} = \frac{1}{m}\Delta^{[\ell]}(A^{[\ell-1]})^T$
4. **Compute** $\frac{\partial L}{\partial b^{[\ell]}} = \frac{1}{m}\Delta^{[\ell]}\mathbf{1}$
5. **Propagate** $\Delta^{[\ell-1]} = (W^{[\ell]})^T\Delta^{[\ell]} \odot g'^{[\ell-1]}(Z^{[\ell-1]})$
6. **If regularised**, add $\frac{\lambda}{m}W^{[\ell]}$ to the weight gradient only.
7. **Update** $W^{[\ell]} \leftarrow W^{[\ell]} - \eta\frac{\partial L_{\text{reg}}}{\partial W^{[\ell]}}$
